# Hybrid Latent-State Language Model - Modular Training

This notebook is a self-contained wrapper around `train_modules.py`, which
trains the latent-state pieces SEPARATELY, each with its own objective
(curriculum: autoencoder 'output sane words' first, then the latent
algebra). It recreates `src/` + `train_modules.py` from embedded cells, then
runs train_modules.py as a subprocess.

**Hypothesis:** decomposing into trainable input->output maps (make_B(A)->B,
continue(A)->A2, Answer_in_format_D(A,B,C)->D, plus token<->state) lets the
latent state actually *contain the correct thing* instead of just learning
to spell tokens.


In [ ]:
import subprocess, sys, os, json, time, random, math
from pathlib import Path

# IMPORTANT: do NOT import torch here. Training runs in a SEPARATE
# subprocess (train_modules.py), so the notebook process only needs the
# right torch installed in the environment. Importing torch in this process
# and then reloading it after a P100 downgrade causes the
# 'Only a single TORCH_LIBRARY can be used to register triton' error.
# Detect the GPU via nvidia-smi and, if it's a P100 (sm_60), install a
# compatible torch into the env; the train_modules.py subprocess loads it.

def gpu_capability():
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
            stderr=subprocess.DEVNULL).decode().strip().splitlines()
        if out:
            return float(out[0].strip())
    except Exception:
        pass
    return None

cap = gpu_capability()
print('GPU compute capability:', cap)
print('Running train_converged.py on CPU (--device cpu): using Kaggle-default torch on CPU; no CUDA, so no P100 torch-downgrade needed.')
print('train_modules.py will report the torch version it loads.')


In [ ]:
# Recreate the reusable src/ package and train_modules.py from the embedded
# cells.
import os
os.makedirs('src', exist_ok=True)
open('src/__init__.py', 'w').close()  # optional, enables `import src...`
print('workspace files ready:', sorted(os.listdir('.')))


In [ ]:
%%writefile src/dataset.py
"""
Toy world generator for synthetic reasoning tasks.

Tasks:
  1. location_tracking - where is X after a long, *interfering* sequence of moves?
  2. inventory_tracking - what does X hold after pickups/drops?
  3. transfer          - multi-hop: where is the item after it changed hands + moved?
  4. exact_recall      - remember a password seen far earlier amid decoys (anti-recency)
  5. story_continuation

Design notes (see AGENTS.md: "too easy / low loss but useless output"):
  * Interfering distractors use REAL entity names and locations, so a model that
    just grabs "the last location word it saw" or attends to the most recent
    mention will be actively misled. This makes low loss require real tracking.
  * Every QA sample is emitted in a strict "Answer: <X>" slot. Evaluation can
    therefore do EXACT MATCH on the generated answer rather than a lazy substring
    check, which is what previously hid useless output behind a low loss.
  * Each sample carries a `meta` dict (difficulty/structure) so evaluation can
    break accuracy down by CONDITION (interference level, recall gap, decoy
    present, multi-hop depth) -- far more informative than a single number.
"""

import random
import json
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional


# --- shared pools ---------------------------------------------------------

FILLER = [
    "It was a quiet day.",
    "The sun was shining.",
    "Time passed slowly.",
    "Nothing unusual happened.",
    "The weather was pleasant.",
    "A soft wind moved through the trees.",
    "The clock on the wall kept ticking.",
    "Shadows stretched long across the floor.",
]

# Literal label used to anchor the answer slot in training and evaluation.
ANSWER_LABEL = "Answer:"


def _interference_sentence(world) -> str:
    """Distractor that mentions a REAL entity/location to create interference."""
    name = random.choice(list(world.entities.keys()))
    loc = random.choice(world.locations)
    templates = [
        f"{name} walked toward the {loc}.",
        f"{name} was seen near the {loc}.",
        f"The {loc} looked different today.",
        f"{name} spent some time in the {loc}.",
        f"Someone mentioned {name} at the {loc}.",
        f"{name} left the {loc} in a hurry.",
    ]
    return random.choice(templates)


@dataclass
class Entity:
    name: str
    location: str
    inventory: List[str] = field(default_factory=list)


@dataclass
class World:
    entities: Dict[str, Entity] = field(default_factory=dict)
    locations: List[str] = field(default_factory=list)
    items: List[str] = field(default_factory=list)
    history: List[str] = field(default_factory=list)
    names: List[str] = field(default_factory=list)

    def __init__(self, names=None, locations=None, items=None):
        self.locations = locations or [
            "kitchen", "bedroom", "garden", "garage", "bathroom",
            "living room", "office", "basement", "attic", "hallway"
        ]
        self.items = items or [
            "apple", "book", "key", "phone", "cup", "pen",
            "wallet", "watch", "bag", "umbrella"
        ]
        self.names = names or [
            "John", "Mary", "Alex", "Sam", "Emma", "Leo",
            "Zoe", "Max", "Lily", "Tom"
        ]
        self.entities = {}
        self.history = []
        self._init_world()

    def _init_world(self):
        names_subset = random.sample(self.names, k=random.randint(3, 5))
        for i, name in enumerate(names_subset):
            loc = random.choice(self.locations)
            if i == 0:
                inv = random.sample(self.items, k=random.randint(1, 2))
            else:
                inv = random.sample(self.items, k=random.randint(0, 2))
            self.entities[name] = Entity(name=name, location=loc, inventory=inv)

    def _narrate(self, sentence: str):
        self.history.append(sentence)

    def move_entity(self, name: str) -> str:
        entity = self.entities[name]
        new_loc = random.choice([l for l in self.locations if l != entity.location])
        old_loc = entity.location
        entity.location = new_loc
        sentence = f"{name} moved from {old_loc} to {new_loc}."
        self._narrate(sentence)
        return sentence

    def pickup_item(self, name: str) -> str:
        entity = self.entities[name]
        available = [i for i in self.items if i not in entity.inventory]
        if not available:
            return ""
        item = random.choice(available)
        entity.inventory.append(item)
        sentence = f"{name} picked up {item}."
        self._narrate(sentence)
        return sentence

    def drop_item(self, name: str) -> str:
        entity = self.entities[name]
        if not entity.inventory:
            return ""
        item = random.choice(entity.inventory)
        entity.inventory.remove(item)
        sentence = f"{name} dropped {item}."
        self._narrate(sentence)
        return sentence

    def give_item(self, name1: str, name2: str) -> str:
        e1 = self.entities[name1]
        e2 = self.entities[name2]
        if not e1.inventory or e1.location != e2.location:
            return ""
        item = random.choice(e1.inventory)
        e1.inventory.remove(item)
        e2.inventory.append(item)
        sentence = f"{name1} gave {item} to {name2}."
        self._narrate(sentence)
        return sentence

    def transfer_item(self, item: str, from_name: str, to_name: str) -> str:
        """Force a transfer (co-locating if needed) and narrate it.

        Used by the multi-hop transfer task where the answer depends on the
        item's final holder's location.
        """
        if item not in self.entities[from_name].inventory:
            return ""
        if self.entities[from_name].location != self.entities[to_name].location:
            self.entities[to_name].location = self.entities[from_name].location
        self.entities[from_name].inventory.remove(item)
        self.entities[to_name].inventory.append(item)
        sentence = f"{from_name} gave the {item} to {to_name}."
        self._narrate(sentence)
        return sentence

    def generate_password(self) -> Tuple[str, str]:
        chars = "ABCDEFGHJKLMNPQRSTUVWXYZ23456789"
        password = "".join(random.choices(chars, k=8))
        entity = random.choice(list(self.entities.keys()))
        sentence = f"The secret code for {entity} is {password}."
        self._narrate(sentence)
        return password, entity

    def _interference(self) -> str:
        return _interference_sentence(self)


def generate_location_task(
    n_moves: int = 10,
    n_interference: int = 8,
    n_filler: int = 3,
    max_chars: int = 600,
) -> Tuple[str, str, str, dict]:
    """Location tracking with interfering mentions of other entities/places."""
    world = World()
    sentences = [
        f"{name} was in the {entity.location}."
        for name, entity in world.entities.items()
    ]

    names = list(world.entities.keys())
    n_interf = 0
    for _ in range(n_moves):
        name = random.choice(names)
        action = random.choice(["move", "pickup", "drop"])
        if action == "move":
            sentences.append(world.move_entity(name))
        elif action == "pickup":
            s = world.pickup_item(name)
            if s:
                sentences.append(s)
        else:
            s = world.drop_item(name)
            if s:
                sentences.append(s)
        if random.random() < 0.5:
            sentences.append(world._interference())
            n_interf += 1
        if len(" ".join(sentences)) > max_chars:
            break

    for _ in range(n_filler):
        sentences.append(random.choice(FILLER))
        if len(" ".join(sentences)) > max_chars:
            break

    narrative = " ".join(s for s in sentences if s)
    target = random.choice(names)
    question = f"Where is {target}?"
    answer = world.entities[target].location
    meta = {"n_moves": n_moves, "n_interference": n_interf}
    return narrative, question, answer, meta


def generate_inventory_task(
    n_actions: int = 6,
    n_interference: int = 4,
    max_chars: int = 600,
) -> Tuple[str, str, str, dict]:
    """Inventory tracking with interfering mentions."""
    world = World()
    sentences = []

    for name, entity in world.entities.items():
        if entity.inventory:
            items_str = ", ".join(entity.inventory)
            sentences.append(f"{name} was in the {entity.location} with {items_str}.")
        else:
            sentences.append(f"{name} was in the {entity.location}.")

    names = list(world.entities.keys())
    n_interf = 0
    for _ in range(n_actions):
        name = random.choice(names)
        action = random.choice(["move", "pickup", "drop"])
        if action == "move":
            sentences.append(world.move_entity(name))
        elif action == "pickup":
            s = world.pickup_item(name)
            if s:
                sentences.append(s)
        else:
            s = world.drop_item(name)
            if s:
                sentences.append(s)
        if random.random() < 0.4:
            sentences.append(world._interference())
            n_interf += 1
        if len(" ".join(sentences)) > max_chars:
            break

    narrative = " ".join(s for s in sentences if s)
    entities_with_items = [n for n in names if world.entities[n].inventory]
    target = random.choice(entities_with_items) if entities_with_items else random.choice(names)
    question = f"What does {target} have?"
    answer = " and ".join(world.entities[target].inventory) if world.entities[target].inventory else "nothing"
    meta = {"n_actions": n_actions, "n_interference": n_interf}
    return narrative, question, answer, meta


def generate_transfer_task(
    n_steps: int = 7,
    n_interference: int = 6,
    max_chars: int = 600,
) -> Tuple[str, str, str, dict]:
    """Multi-hop: track an item as it changes hands and its holder moves.

    The answer (the item's final location) requires combining two reasoning
    steps: who currently holds the item, and where that holder ended up.
    """
    world = World()
    sentences = [
        f"{name} was in the {entity.location}."
        for name, entity in world.entities.items()
    ]

    names = list(world.entities.keys())
    item = random.choice(world.items)
    holder = random.choice(names)
    world.entities[holder].inventory.append(item)
    sentences.append(f"{holder} had the {item}.")

    n_interf = 0
    for _ in range(n_steps):
        r = random.random()
        if r < 0.45:
            recip = random.choice([n for n in names if n != holder])
            s = world.transfer_item(item, holder, recip)
            if s:
                sentences.append(s)
                holder = recip
            else:
                sentences.append(world.move_entity(holder))
        else:
            sentences.append(world.move_entity(holder))
        if random.random() < 0.5:
            sentences.append(world._interference())
            n_interf += 1
        if len(" ".join(sentences)) > max_chars:
            break

    narrative = " ".join(s for s in sentences if s)
    question = f"Where is the {item}?"
    answer = world.entities[holder].location
    meta = {"n_hops": n_steps, "n_interference": n_interf}
    return narrative, question, answer, meta


def generate_recall_task(
    n_distractor_sentences: int = 40,
    decoy_for_target: bool = True,
    max_chars: int = 600,
) -> Tuple[str, str, str, dict]:
    """Exact recall with a long gap, decoy codes, and an anti-recency trap.

    The target code is shown early. The distractor stream contains many other
    codes (for other entities) plus interfering filler. If decoy_for_target is
    set, the SAME entity's code is later "updated", and the question asks for
    the FIRST code -- so a model relying on recency will be wrong.

    Distractors are added until the narrative reaches `max_chars`, which keeps
    the whole sample (narrative + question + answer) inside a small char-level
    context window instead of being silently truncated.
    """
    world = World()
    sentences = [
        f"{name} was in the {entity.location}."
        for name, entity in world.entities.items()
    ]

    password, entity_name = world.generate_password()
    sentences.append(f"The secret code for {entity_name} is {password}.")

    other_entities = [n for n in world.entities if n != entity_name]
    n_added = 0
    for _ in range(n_distractor_sentences):
        r = random.random()
        if r < 0.3 and other_entities:
            p2, e2 = world.generate_password()
            sentences.append(f"The secret code for {e2} is {p2}.")
        elif r < 0.6:
            sentences.append(world._interference())
        else:
            sentences.append(random.choice(FILLER))
        n_added += 1
        if len(" ".join(sentences)) > max_chars:
            break

    gap_chars = len(" ".join(sentences))  # distance from code to question

    if decoy_for_target:
        chars = "ABCDEFGHJKLMNPQRSTUVWXYZ23456789"
        decoy = "".join(random.choices(chars, k=8))
        sentences.append(f"The secret code for {entity_name} was updated to {decoy}.")
        question = f"What was the FIRST secret code for {entity_name}?"
    else:
        question = f"What is the secret code for {entity_name}?"

    answer = password
    narrative = " ".join(s for s in sentences if s)
    meta = {
        "n_distractors": n_added,
        "gap_chars": gap_chars,
        "has_decoy": decoy_for_target,
    }
    return narrative, question, answer, meta


def generate_story_prompt(
    n_setup_sentences: int = 3,
) -> Tuple[str, str, dict]:
    """Story continuation prompt + ground-truth opening line."""
    world = World()
    sentences = []

    name = list(world.entities.keys())[0]
    entity = world.entities[name]

    sentences.append(f"Write a story about {name}.")
    sentences.append(f"{name} was in the {entity.location}.")

    for _ in range(n_setup_sentences - 1):
        actions = [
            f"{name} looked around.",
            f"{name} felt something was about to happen.",
            f"Something caught {name}'s attention.",
            f"{name} heard a strange noise.",
            f"The air felt different.",
        ]
        sentences.append(random.choice(actions))

    prompt = " ".join(sentences)

    continuations = [
        f"Suddenly, {name} noticed something unusual.",
        f"Then {name} decided to explore further.",
        f"Out of nowhere, a mysterious figure appeared.",
        f"At that moment, everything changed.",
    ]
    ground_truth = random.choice(continuations)
    meta = {}
    return prompt, ground_truth, meta


# --- formatting helpers ---------------------------------------------------
# These guarantee train/eval use the SAME surface form so the model learns to
# emit its answer in the "Answer:" slot, enabling strict exact-match scoring.

def format_for_training(sample: dict) -> str:
    """Full text the model is trained on (narrative + question + answer)."""
    if sample["task_type"] == "story":
        return f"{sample['narrative']}\n{sample['answer']}"
    return (
        f"{sample['narrative']}\n"
        f"Question: {sample['question']}\n"
        f"{ANSWER_LABEL} {sample['answer']}"
    )


def build_prompt(sample: dict) -> str:
    """Prompt fed at inference: narrative + question + 'Answer: '.

    Ends with a trailing space so the surface form exactly matches
    format_for_training (which writes 'Answer: <ans>'); otherwise the model
    sees a context at inference it never encountered during training.
    """
    if sample["task_type"] == "story":
        return sample["narrative"] + "\n"
    return (
        f"{sample['narrative']}\n"
        f"Question: {sample['question']}\n"
        f"{ANSWER_LABEL} "
    )


def parse_answer(generated: str) -> str:
    """Extract the predicted answer from generated text (before any newline)."""
    text = generated.strip()
    if "\n" in text:
        text = text.split("\n", 1)[0].strip()
    if text.lower().startswith(ANSWER_LABEL.lower()):
        text = text[len(ANSWER_LABEL):].strip()
    return text


def _bucket(sample: dict) -> str:
    """Coarse difficulty bucket for stratified accuracy reporting."""
    t = sample["task_type"]
    m = sample.get("meta", {})
    if t in ("location", "inventory", "transfer"):
        ni = m.get("n_interference", 0)
        if ni == 0:
            return "interf=0"
        if ni <= 2:
            return "interf=1-2"
        return "interf>=3"
    if t == "recall":
        if m.get("has_decoy"):
            return "decoy=yes"
        return "decoy=no"
    return "all"


def generate_dataset(
    n_samples: int = 1000,
    seed: int = 42,
    task_weights: Optional[Dict[str, float]] = None,
    location_max_chars: int = 600,
    inventory_max_chars: int = 600,
    transfer_max_chars: int = 600,
    recall_max_chars: int = 600,
) -> List[Dict]:
    """Generate a mixed dataset of reasoning tasks.

    `*-max-chars` controls narrative length per task type (so the longest
    sample still fits the model's context window). For a quick CPU run you can
    pass smaller values (e.g. recall_max_chars=400) to keep context short.
    """
    random.seed(seed)

    if task_weights is None:
        task_weights = {
            "location": 0.3,
            "inventory": 0.2,
            "transfer": 0.2,
            "recall": 0.2,
            "story": 0.1,
        }

    tasks = list(task_weights.keys())
    weights = list(task_weights.values())

    dataset = []
    for _ in range(n_samples):
        task_type = random.choices(tasks, weights=weights, k=1)[0]

        if task_type == "location":
            narrative, question, answer, meta = generate_location_task(max_chars=location_max_chars)
        elif task_type == "inventory":
            narrative, question, answer, meta = generate_inventory_task(max_chars=inventory_max_chars)
        elif task_type == "transfer":
            narrative, question, answer, meta = generate_transfer_task(max_chars=transfer_max_chars)
        elif task_type == "recall":
            narrative, question, answer, meta = generate_recall_task(max_chars=recall_max_chars)
        elif task_type == "story":
            narrative, answer, meta = generate_story_prompt()
            question = ""
        else:
            continue

        dataset.append({
            "narrative": narrative,
            "question": question,
            "answer": answer,
            "task_type": task_type,
            "meta": meta,
        })

    return dataset


if __name__ == "__main__":
    dataset = generate_dataset(n_samples=8, seed=42)
    for i, sample in enumerate(dataset):
        print(f"\n=== Sample {i+1} [{sample['task_type']}] meta={sample['meta']} ===")
        print(f"Narrative: {sample['narrative']}")
        if sample["question"]:
            print(f"Question: {sample['question']}")
        print(f"Answer: {sample['answer']}")
        print(f"-- train text --\n{format_for_training(sample)}")


In [ ]:
%%writefile src/tokenizer.py
"""
Simple character-level tokenizer for synthetic tasks.

Since we're generating our own text, we don't need a heavy tokenizer.
Character-level is sufficient for the toy world tasks and keeps vocab small.
"""

from collections import Counter
from typing import List, Dict, Optional, Tuple
import json
import os


class CharTokenizer:
    """Character-level tokenizer with special tokens."""

    def __init__(
        self,
        texts: Optional[List[str]] = None,
        max_vocab: int = 256,
        pad_token: str = "<PAD>",
        unk_token: str = "<UNK>",
        eos_token: str = "<EOS>",
        sep_token: str = "<SEP>",
    ):
        self.pad_token = pad_token
        self.unk_token = unk_token
        self.eos_token = eos_token
        self.sep_token = sep_token

        # Build vocabulary
        self.char_counts = Counter()
        if texts:
            for text in texts:
                self.char_counts.update(text)

        # Special tokens first
        self.special_tokens = [pad_token, unk_token, eos_token, sep_token]
        self.vocab = {c: idx + len(self.special_tokens) for idx, (c, _) in enumerate(self.char_counts.most_common(max_vocab - len(self.special_tokens)))}
        for i, tok in enumerate(self.special_tokens):
            self.vocab[tok] = i

        self.inv_vocab = {i: c for c, i in self.vocab.items()}
        self.vocab_size = len(self.vocab)

    def encode(self, text: str, max_len: Optional[int] = None) -> List[int]:
        """Encode text to token IDs."""
        ids = [self.vocab.get(c, self.vocab[self.unk_token]) for c in text]
        if max_len:
            ids = ids[:max_len]
        return ids

    def decode(self, ids: List[int]) -> str:
        """Decode token IDs to text."""
        return "".join(self.inv_vocab.get(i, self.unk_token) for i in ids)

    def save(self, path: str):
        """Save vocabulary to JSON."""
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        with open(path, 'w') as f:
            json.dump({
                "vocab": self.vocab,
                "special_tokens": self.special_tokens,
                "vocab_size": self.vocab_size,
            }, f)

    @classmethod
    def load(cls, path: str) -> "CharTokenizer":
        """Load vocabulary from JSON."""
        with open(path) as f:
            data = json.load(f)
        tok = cls()
        tok.vocab = data["vocab"]
        tok.special_tokens = data["special_tokens"]
        tok.inv_vocab = {i: c for c, i in tok.vocab.items()}
        tok.vocab_size = len(tok.vocab)
        return tok


def build_tokenizer_from_dataset(dataset: List[dict], max_vocab: int = 256) -> CharTokenizer:
    """Build tokenizer from dataset samples.

    Includes the QA formatting markers (Question:, Answer:, newline, ':') so
    they are never mapped to <UNK> when the strict "Answer: <X>" surface form
    is used during training/evaluation.
    """
    texts = []
    for sample in dataset:
        texts.append(sample["narrative"])
        if sample.get("question"):
            texts.append(sample["question"])
        texts.append(sample["answer"])
    # Ensure format markers are present in the vocabulary.
    texts += ["Question:", "Answer:", "\n", ":", "The secret code for was updated to"]
    return CharTokenizer(texts, max_vocab=max_vocab)


In [ ]:
%%writefile src/models.py
"""
Model architectures for the hybrid latent-state language model.

Models:
  - BaselineTransformer: standard autoregressive transformer
  - LatentSSM: recurrent latent model with thinking loop (causal LM)
  - LatentSSMDecoder: SSM + FFN decoder (cheap multi-token generation)

Key design: All models produce [batch, seq_len, vocab_size] output for
fair comparison on next-token prediction.

The latent models additionally perform N "thinking" steps in latent space
between processing input tokens, testing whether extra latent computation
improves reasoning over pure token-by-token processing.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple


# ============================================================
# Baseline Transformer
# ============================================================

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding, generated on the fly for any length."""
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.d_model = d_model
        self._cache: dict = {}

    def forward(self, x):
        seq_len = x.size(1)
        device = x.device
        if seq_len not in self._cache or self._cache[seq_len].device != device:
            position = torch.arange(0, seq_len, dtype=torch.float, device=device).unsqueeze(1)
            div_term = torch.exp(
                torch.arange(0, self.d_model, 2, dtype=torch.float, device=device)
                * (-math.log(10000.0) / self.d_model)
            )
            pe = torch.zeros(seq_len, self.d_model, device=device)
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            self._cache[seq_len] = pe.unsqueeze(0)
        return self.dropout(x + self._cache[seq_len])


class BaselineTransformer(nn.Module):
    """
    Standard autoregressive transformer baseline.

    Architecture:
      tokens -> embedding + positional encoding -> transformer layers -> logits

    Output: [batch, seq_len, vocab_size]
    """

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 4,
        dim_ff: int = 512,
        dropout: float = 0.1,
        max_seq_len: int = 512,
    ):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(d_model, vocab_size)

    def forward(self, src: torch.Tensor) -> torch.Tensor:
        """
        Args:
            src: [batch, seq_len] token IDs
        Returns:
            logits: [batch, seq_len, vocab_size]
        """
        mask = self._generate_square_subsequent_mask(src.size(1), src.device)
        x = self.embedding(src) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        x = self.transformer(x, mask=mask)
        logits = self.output_proj(x)
        return logits

    @staticmethod
    def _generate_square_subsequent_mask(sz: int, device: torch.device) -> torch.Tensor:
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask


# ============================================================
# SSM Layer (simplified Mamba-style)
# ============================================================

class SSMLayer(nn.Module):
    """
    State Space Model layer with input-dependent dynamics (Mamba-style).

    Uses a recurrent update: h_t = A(x_t) * h_{t-1} + B(x_t) * x_t
    where A and B are functions of the input (selective mechanism).

    This allows the model to selectively remember/forget based on input.
    """

    def __init__(self, d_state: int, d_input: int, selective: bool = True):
        super().__init__()
        self.d_state = d_state
        self.d_input = d_input
        self.selective = selective

        # Base state transition matrix
        self.A_base = nn.Parameter(torch.randn(d_state, d_state) * 0.1)
        
        if selective:
            # Input-dependent modulation of A
            self.A_mod = nn.Linear(d_input, d_state * d_state, bias=False)
            # Input-dependent B
            self.B = nn.Linear(d_input, d_state, bias=False)
        else:
            # Simple fixed A and B
            self.A = self.A_base
            self.B = nn.Linear(d_input, d_state)

        # Output projection
        self.C = nn.Linear(d_state, d_state)

        # Nonlinearity
        self.act = nn.SiLU()

        # Normalization
        self.norm = nn.LayerNorm(d_state)

    def forward(self, x: torch.Tensor, h: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Process a sequence of inputs.

        Args:
            x: [batch, seq_len, d_input] input
            h: [batch, d_state] initial state (optional)
        Returns:
            output: [batch, seq_len, d_state]
            h_final: [batch, d_state] final state
        """
        batch, seq_len, _ = x.shape
        if h is None:
            h = torch.zeros(batch, self.d_state, device=x.device)

        outputs = []
        for t in range(seq_len):
            h = self.step(x[:, t, :], h)
            outputs.append(h)

        outputs = torch.stack(outputs, dim=1)  # [batch, seq_len, d_state]
        return outputs, h

    def step(self, x: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        """
        Single recurrent step with optional input-dependent dynamics.

        Args:
            x: [batch, d_input] input at current step
            h: [batch, d_state] current state
        Returns:
            h_new: [batch, d_state] updated state
        """
        if self.selective:
            # Compute input-dependent A: A(x) = A_base + A_mod(x)
            batch_size = x.size(0)
            A_delta = self.A_mod(x).view(batch_size, self.d_state, self.d_state)
            # Small modulation to keep stable
            A = self.A_base.unsqueeze(0) + 0.1 * torch.tanh(A_delta)
            # State update: h_new = A @ h + B(x)
            h = torch.bmm(A, h.unsqueeze(-1)).squeeze(-1) + self.B(x)
        else:
            # Simple linear update
            h = torch.einsum('ij,bj->bi', self.A_base, h) + self.B(x)
        
        h = self.act(h)
        h = self.norm(h)
        return h


# ============================================================
# Latent SSM Model (Causal LM with thinking steps)
# ============================================================

class LatentSSM(nn.Module):
    """
    Recurrent latent model with thinking loop.

    Architecture:
      For each input token:
        1. Embed token
        2. Update SSM state
        3. Every `think_every` tokens: do N latent thinking steps
        4. Decode state -> logits for this position

    This produces [batch, seq_len, vocab_size] output, same as the baseline,
    enabling fair comparison. The latent model does extra computation
    (thinking steps) between token predictions.

    Output: [batch, seq_len, vocab_size]
    """

    def __init__(
        self,
        vocab_size: int,
        d_state: int = 256,
        d_model: int = 256,
        num_ssm_layers: int = 2,
        latent_steps: int = 4,
        think_every: int = 1,
        dropout: float = 0.1,
        selective: bool = True,
    ):
        super().__init__()
        self.d_state = d_state
        self.d_model = d_model
        self.latent_steps = latent_steps
        self.think_every = think_every

        self.embedding = nn.Embedding(vocab_size, d_model)

        # Input projection (token embedding -> SSM input)
        self.input_proj = nn.Linear(d_model, d_state)

        # Stack of SSM layers
        self.ssm_layers = nn.ModuleList([
            SSMLayer(d_state, d_state, selective=selective) for _ in range(num_ssm_layers)
        ])

        # FFN for latent thinking
        self.ffn = nn.Sequential(
            nn.Linear(d_state, d_state * 2),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_state * 2, d_state),
        )

        # Token decoder (from state to vocab)
        self.token_decoder = nn.Linear(d_state, vocab_size)

        # Residual gating for thinking steps
        self.gate = nn.Sequential(
            nn.Linear(d_state, d_state),
            nn.Sigmoid(),
        )

    def think(self, h: torch.Tensor, steps: Optional[int] = None) -> torch.Tensor:
        """
        Run latent thinking steps without consuming input tokens.

        Args:
            h: [batch, d_state] current state
            steps: number of thinking steps (default: self.latent_steps)
        Returns:
            h: [batch, d_state] updated state
        """
        steps = steps or self.latent_steps
        for _ in range(steps):
            h_new = h
            for ssm in self.ssm_layers:
                # Self-recurrence: use state as both input and state
                h_new = ssm.step(h_new, h_new)
            h_new = self.ffn(h_new)
            # Gated residual
            gate = self.gate(h)
            h = gate * h_new + (1 - gate) * h
        return h

    def forward(self, src: torch.Tensor) -> torch.Tensor:
        """
        Full forward pass: process tokens sequentially with optional thinking.

        Args:
            src: [batch, seq_len] token IDs
        Returns:
            logits: [batch, seq_len, vocab_size]
        """
        batch, seq_len = src.shape
        device = src.device

        # Initialize state
        h = torch.zeros(batch, self.d_state, device=device)

        # Embed all tokens
        x = self.embedding(src)  # [batch, seq, d_model]
        x = self.input_proj(x)   # [batch, seq, d_state]

        all_logits = []

        for t in range(seq_len):
            # Update SSM state with current token
            for ssm in self.ssm_layers:
                h = ssm.step(x[:, t, :], h)

            # Periodic latent thinking
            if self.think_every > 0 and (t + 1) % self.think_every == 0:
                h = self.think(h)

            # Decode to logits
            logits_t = self.token_decoder(h)  # [batch, vocab_size]
            all_logits.append(logits_t)

        logits = torch.stack(all_logits, dim=1)  # [batch, seq_len, vocab_size]
        return logits


# ============================================================
# Latent SSM + Decoder (cheap multi-token generation)
# ============================================================

class LatentSSMDecoder(nn.Module):
    """
    SSM + FFN decoder for multi-token generation.

    Key difference from LatentSSM:
    - After thinking, can generate multiple tokens cheaply from same state
    - Decoder uses positional embeddings to generate tokens_per_step tokens
    - State evolves slowly, tokens generated quickly

    Training mode: produces [batch, seq_len, vocab_size] for fair comparison
    Generation mode: can produce multiple tokens per state update

    Output: [batch, seq_len, vocab_size]
    """

    def __init__(
        self,
        vocab_size: int,
        d_state: int = 256,
        d_model: int = 256,
        num_ssm_layers: int = 2,
        latent_steps: int = 4,
        tokens_per_step: int = 8,
        think_every: int = 4,
        dropout: float = 0.1,
        selective: bool = True,
    ):
        super().__init__()
        self.d_state = d_state
        self.d_model = d_model
        self.latent_steps = latent_steps
        self.tokens_per_step = tokens_per_step
        self.think_every = think_every

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.input_proj = nn.Linear(d_model, d_state)

        # SSM layers
        self.ssm_layers = nn.ModuleList([
            SSMLayer(d_state, d_state, selective=selective) for _ in range(num_ssm_layers)
        ])

        # FFN for latent thinking
        self.ffn = nn.Sequential(
            nn.Linear(d_state, d_state * 2),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_state * 2, d_state),
        )

        # Token decoder with position encoding
        self.decoder_ffn = nn.Sequential(
            nn.Linear(d_state + 32, d_state),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_state, vocab_size),
        )
        self.pos_embed = nn.Parameter(torch.randn(tokens_per_step, 32))

        # Gating
        self.gate = nn.Sequential(
            nn.Linear(d_state, d_state),
            nn.Sigmoid(),
        )

    def think(self, h: torch.Tensor, steps: Optional[int] = None) -> torch.Tensor:
        """Run latent thinking steps."""
        steps = steps or self.latent_steps
        for _ in range(steps):
            h_new = h
            for ssm in self.ssm_layers:
                h_new = ssm.step(h_new, h_new)
            h_new = self.ffn(h_new)
            gate = self.gate(h)
            h = gate * h_new + (1 - gate) * h
        return h

    def decode_tokens(self, h: torch.Tensor, n_tokens: int) -> torch.Tensor:
        """
        Generate multiple tokens from a single state.

        Args:
            h: [batch, d_state] state
            n_tokens: number of tokens to generate
        Returns:
            logits: [batch, n_tokens, vocab_size]
        """
        n_tokens = min(n_tokens, self.tokens_per_step)
        batch = h.size(0)

        h_expanded = h.unsqueeze(1).expand(-1, n_tokens, -1)
        pos = self.pos_embed[:n_tokens].unsqueeze(0).expand(batch, -1, -1)

        x = torch.cat([h_expanded, pos], dim=-1)
        logits = self.decoder_ffn(x)
        return logits

    def forward(self, src: torch.Tensor) -> torch.Tensor:
        """
        Full forward pass for training.

        Process tokens sequentially, periodically think, and decode
        multiple tokens per state. For fair comparison with baseline,
        produces [batch, seq_len, vocab_size].

        Args:
            src: [batch, seq_len] token IDs
        Returns:
            logits: [batch, seq_len, vocab_size]
        """
        batch, seq_len = src.shape
        device = src.device

        # Initialize state
        h = torch.zeros(batch, self.d_state, device=device)

        # Embed all tokens
        x = self.embedding(src)
        x = self.input_proj(x)

        all_logits = []
        token_idx = 0  # How many output tokens we've produced from current state

        for t in range(seq_len):
            # Update SSM state with current token
            for ssm in self.ssm_layers:
                h = ssm.step(x[:, t, :], h)

            # Periodic latent thinking
            if self.think_every > 0 and (t + 1) % self.think_every == 0:
                h = self.think(h)
                token_idx = 0  # Reset token counter after thinking

            # Decode from current state
            if token_idx < self.tokens_per_step:
                logits_t = self.decode_tokens(h, 1)  # [batch, 1, vocab]
                logits_t = logits_t.squeeze(1)  # [batch, vocab]
                token_idx += 1
            else:
                # Just decode directly without positional bias
                logits_t = self.decode_tokens(h, 1).squeeze(1)

            all_logits.append(logits_t)

        logits = torch.stack(all_logits, dim=1)  # [batch, seq_len, vocab_size]
        return logits


In [ ]:
%%writefile src/trainer.py
"""
Training loop for latent-state language model experiments.

Handles:
  - Training with multiple loss types
  - Experiment tracking
  - Checkpointing
  - Evaluation
  - Sample generation
"""

import os
import json
import time
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

try:
    from src.dataset import build_prompt, parse_answer, _bucket
except ImportError:  # allow running trainer standalone
    from dataset import build_prompt, parse_answer, _bucket


@dataclass
class ExperimentConfig:
    """Configuration for a single experiment."""
    exp_id: str = "exp001"
    model: str = "baseline"  # baseline, latent_ssm, latent_ssm_decoder

    # Model params
    d_model: int = 256
    d_state: int = 256
    nhead: int = 8
    num_layers: int = 4
    num_ssm_layers: int = 2
    latent_steps: int = 4
    tokens_per_step: int = 8

    # Training params
    batch_size: int = 32
    learning_rate: float = 3e-4
    num_epochs: int = 20
    max_seq_len: int = 256
    gradient_clip: float = 1.0
    warmup_steps: int = 100

    # Loss weights
    token_loss_weight: float = 1.0
    answer_loss_weight: float = 0.0   # 0 = standard next-token loss over all chars
                                       # >0 = up-weight loss on the answer-slot tokens
                                       #      so the model gets sharp signal on what to fill
                                       #      Reward *only* comes from predicting the right
                                       #      answer chars, not just fitting filler.
    latent_consistency_weight: float = 0.1
    state_evolution_weight: float = 0.1

    # Visibility
    print_every_batches: int = 50     # visible "TRAIN" log every N batches with loss + a sample
    gen_sample_every: int = 200       # generate a sample every N batches (heavy)
    save_initial_loss_baseline: bool = True

    # Misc
    seed: int = 42
    device: str = "auto"
    save_every: int = 5
    eval_every: int = 1

    def to_dict(self) -> dict:
        return {k: v for k, v in self.__dict__.items()}

    @classmethod
    def from_dict(cls, d: dict) -> "ExperimentConfig":
        return cls(**{k: v for k, v in d.items() if k in cls.__dataclass_fields__})


class TextDataset(Dataset):
    """Simple text dataset for training."""

    def __init__(self, texts: List[str], tokenizer, max_len: int = 256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        ids = self.tokenizer.encode(text, max_len=self.max_len)
        ids = ids + [self.tokenizer.vocab[self.tokenizer.eos_token]]
        ids = ids[:self.max_len + 1]

        # Pad to max_len + 1
        while len(ids) < self.max_len + 1:
            ids.append(self.tokenizer.vocab[self.tokenizer.pad_token])

        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        return x, y


class Trainer:
    """Training loop with experiment tracking and checkpointing."""

    def __init__(self, model: nn.Module, config: ExperimentConfig, tokenizer, output_dir: str = "experiments"):
        self.model = model
        self.config = config
        self.tokenizer = tokenizer
        self.output_dir = Path(output_dir) / config.exp_id
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # Device
        if config.device == "auto":
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(config.device)
        self.model = self.model.to(self.device)

        # Optimizer
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config.learning_rate,
            weight_decay=0.01,
        )

        # Scheduler
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimizer,
            max_lr=config.learning_rate,
            total_steps=config.num_epochs * 100,  # approximate
            pct_start=0.1,
        )

        # Metrics
        self.metrics = {
            "train_loss": [],
            "val_loss": [],
            "eval_accuracy": [],
            "samples": [],
        }

    def _answer_positions(self, ids: torch.Tensor) -> Optional[torch.Tensor]:
        """Find the token positions right AFTER 'Answer: ' in each batch row.

        Returns a [batch, seq_len] bool mask (True at the answer positions)
        OR None if 'Answer:' isn't a contiguous pattern in the vocab (then
        we silently fall back to plain next-token loss).

        Why focus on the answer slot?  Most training characters are easy filler
        (spaces, the name John, the verb "moved"). Gradient from those drowns
        out the signal for "what goes in the answer slot." Focusing loss on
        the answer slot means the model gets a sharp, focused signal that
        directly improves exact-match accuracy.
        """
        ans_lit = self.tokenizer.encode("Answer: ", max_len=None)
        if len(ans_lit) < 3:
            return None  # can't pattern-match
        L = len(ans_lit)
        bsz, sl = ids.shape
        if sl <= L:
            return None
        mask = torch.zeros((bsz, sl), dtype=torch.bool, device=ids.device)
        # Slide a window per row.
        ids_cpu = ids.detach()
        for b in range(bsz):
            row = ids_cpu[b].tolist()
            # Build rolling "last L tokens equal ans_lit"
            i = L - 1
            in_run = False
            for t in range(L, sl):
                if row[t - L:t] == ans_lit:
                    # tokens indices [t, t+1, ...] are answer positions
                    # but we want the *target* positions (y = ids[1:]), which
                    # are answer positions [t-1, t, t+1, ...]. Align with y:
                    # y[t-1] is the first answer char.
                    mask[b, max(0, t - 1):] = True
                    break
        return mask

    def train_epoch(self, dataloader: DataLoader, eval_during_epoch: bool = False) -> Tuple[float, dict]:
        """Run one training epoch with mid-epoch visibility.

        Two loss numbers are reported mid-epoch so the user can SEE whether
        the model is improving on filler (loss_full) or on the answer slot
        (loss_answer) specifically. Modest mid-epoch generation + cheap QA
        snapshots make training legible instead of "just a number at the end."
        """
        self.model.train()
        total_loss = 0
        n_batches = 0
        info = {"mid_epoch_samples": [], "mid_epoch_qa": []}

        pb_every = self.config.print_every_batches
        g_every   = self.config.gen_sample_every

        pbar = tqdm(dataloader, desc=f"Training [{self.config.exp_id}]")
        for batch_x, batch_y in pbar:
            batch_x = batch_x.to(self.device)
            batch_y = batch_y.to(self.device)

            self.optimizer.zero_grad()

            # Forward pass - all models produce [batch, seq, vocab]
            logits = self.model(batch_x)
            pad_id = self.tokenizer.vocab[self.tokenizer.pad_token]

            # Build per-token loss. We compute the answer-slot mask *over the
            # input positions* (batch_x) but use it with the target positions
            # (batch_y) by sliding one step earlier.
            flat_logits = logits.reshape(-1, logits.size(-1))
            flat_targets = batch_y.reshape(-1)
            loss_per_token = F.cross_entropy(
                flat_logits, flat_targets, ignore_index=pad_id, reduction='none',
            ).reshape(batch_y.shape)

            # Standard uniform cross-entropy weight is 1.0. Optional answer-focus:
            # weight = 1 + answer_loss_weight * mask_of_answer_positions. Default 0.
            ans_w = self.config.answer_loss_weight
            if ans_w > 0:
                mask_in = self._answer_positions(batch_x)
                if mask_in is not None:
                    # align mask with batch_y: answer chars in batch_y are
                    # shifted one position earlier than in batch_x.
                    mask = torch.zeros_like(mask_in)
                    mask[:, :-1] = mask_in[:, 1:]  # y[:, t] is x[:, t+1]
                else:
                    mask = torch.zeros_like(batch_y, dtype=torch.bool)
                mask = mask & (batch_y != pad_id)  # don't double-count pad
                weights = 1.0 + ans_w * mask.to(loss_per_token.dtype)
            else:
                weights = torch.ones_like(loss_per_token)

            non_pad = (batch_y != pad_id).float()
            loss = (loss_per_token * weights * non_pad).sum() / non_pad.sum().clamp(min=1.0)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.gradient_clip)
            self.optimizer.step()
            self.scheduler.step()

            total_loss += loss.item()
            n_batches += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

            # -------- visible TRAIN progress --------
            if pb_every > 0 and (n_batches % pb_every == 0):
                with torch.no_grad():
                    if ans_w > 0 and mask.any():
                        ans_l = (loss_per_token * mask.to(loss_per_token.dtype)).sum() / mask.float().sum().clamp(min=1.0)
                    else:
                        ans_l = loss_per_token.mean()
                    full_l = loss_per_token.mean()
                print(f"  [TRAIN] exp={self.config.exp_id} ep={len(self.metrics['train_loss'])+1} "
                      f"batch={n_batches}/{len(dataloader)} "
                      f"loss_full={full_l.item():.3f} loss_answer={ans_l.item():.3f} "
                      f"lr={self.optimizer.param_groups[0]['lr']:.2e}")

            # -------- visible mid-epoch generation ----------
            if eval_during_epoch and g_every > 0 and (n_batches % g_every == 0):
                snap = self._mid_epoch_sample()
                if snap:
                    info["mid_epoch_samples"].append(snap)

            # -------- mid-epoch cheap QA ----------
            if eval_during_epoch and g_every > 0 and (n_batches % (2 * g_every) == 0):
                qa = self.run_qa_eval_quick()
                if qa:
                    info["mid_epoch_qa"].append(qa)
                    print(f"  [MID-QA] n_eval={qa['n']} acc={qa['overall_accuracy']:.3f}")

        return total_loss / max(n_batches, 1), info

    @torch.no_grad()
    def _mid_epoch_sample(self) -> Optional[dict]:
        """Generate one sample mid-epoch so we can see what the model is doing."""
        from src.dataset import build_prompt, parse_answer
        ds = getattr(self, "_qa_snapshot_dataset", None)
        if not ds:
            return None
        sample = ds[0]
        if not sample.get("question"):
            return None
        prompt = build_prompt(sample)
        # greedy + a tiny chance of sampling is bad; use greedy for visibility
        gen = self.generate(prompt, max_tokens=24, temperature=0.0)
        print(f"  [GEN ] Q={sample['question']!r} "
              f"expected={sample['answer']!r} got={gen.strip()!r}")
        return {"prompt": prompt, "expected": sample["answer"], "generated": gen.strip()}

    @torch.no_grad()
    def run_qa_eval_quick(self, n: int = 32) -> Optional[dict]:
        """Cheap, fast QA on first n val samples for live-monitoring."""
        from src.dataset import build_prompt, parse_answer
        ds = getattr(self, "_qa_snapshot_dataset", None)
        if not ds:
            return None
        self.model.eval()
        ok, tot = 0, 0
        for s in ds[:n]:
            if not s.get("question"):
                continue
            prompt = build_prompt(s)
            gen = self.generate(prompt, max_tokens=20, temperature=0.0)
            pred = parse_answer(gen).strip().lower()
            exp = s["answer"].strip().lower()
            ok += int(pred == exp); tot += 1
        return {"overall_accuracy": ok / max(tot, 1), "n": tot}

    @torch.no_grad()
    def evaluate(self, dataloader: DataLoader) -> float:
        """Run evaluation."""
        self.model.eval()
        total_loss = 0
        n_batches = 0

        for batch_x, batch_y in dataloader:
            batch_x = batch_x.to(self.device)
            batch_y = batch_y.to(self.device)

            logits = self.model(batch_x)
            logits = logits.reshape(-1, logits.size(-1))
            targets = batch_y.reshape(-1)
            
            loss = F.cross_entropy(logits, targets, ignore_index=self.tokenizer.vocab[self.tokenizer.pad_token])
            total_loss += loss.item()
            n_batches += 1

        return total_loss / max(n_batches, 1)

    @torch.no_grad()
    def generate(self, prompt: str, max_tokens: int = 50, temperature: float = 0.8, top_k: int = 40) -> str:
        """Generate text from a prompt.

        temperature <= 0 disables sampling and uses greedy argmax (used for
        deterministic, exact-match evaluation). Generation stops at EOS or a
        newline so we capture just the answer in the "Answer:" slot.
        """
        self.model.eval()

        prompt_ids = self.tokenizer.encode(prompt, max_len=self.config.max_seq_len)
        generated = list(prompt_ids)

        eos_id = self.tokenizer.vocab[self.tokenizer.eos_token]
        newline_id = self.tokenizer.vocab.get("\n", None)

        for _ in range(max_tokens):
            x = torch.tensor([generated], dtype=torch.long).to(self.device)
            logits = self.model(x)
            next_logits = logits[0, -1, :]

            if temperature <= 0:
                next_token = int(torch.argmax(next_logits).item())
            else:
                next_logits = next_logits / temperature
                if top_k > 0:
                    kth = torch.topk(next_logits, top_k)[0][..., -1, None]
                    next_logits[next_logits < kth] = float('-inf')
                probs = F.softmax(next_logits, dim=-1)
                next_token = int(torch.multinomial(probs, num_samples=1).item())

            if next_token == eos_id:
                break
            if newline_id is not None and next_token == newline_id:
                break
            generated.append(next_token)

        return self.tokenizer.decode(generated[len(prompt_ids):])

    @torch.no_grad()
    def run_qa_eval(self, dataset: List[dict], max_new_tokens: int = 20) -> Dict[str, float]:
        """Strict exact-match evaluation on QA tasks, with stratified breakdowns.

        Returns overall accuracy, per-task accuracy, and accuracy stratified by
        difficulty bucket (interference level / decoy presence / etc.) so we can
        see *where* the model fails -- not just a single number. This also
        exposes "low loss but useless output" that the old substring check hid.
        """
        self.model.eval()
        correct = 0
        total = 0
        by_task = {}
        by_bucket = {}

        for sample in dataset:
            if not sample.get("question"):  # skip story tasks
                continue

            task_type = sample["task_type"]
            by_task.setdefault(task_type, {"correct": 0, "total": 0})
            bucket = _bucket(sample)
            bkey = f"{task_type}|{bucket}"
            by_bucket.setdefault(bkey, {"correct": 0, "total": 0})

            prompt = build_prompt(sample)
            expected = sample["answer"].strip().lower()

            generated = self.generate(prompt, max_tokens=max_new_tokens, temperature=0.0)
            predicted = parse_answer(generated).strip().lower()

            ok = (predicted == expected)
            if ok:
                correct += 1
                by_task[task_type]["correct"] += 1
                by_bucket[bkey]["correct"] += 1
            total += 1
            by_task[task_type]["total"] += 1
            by_bucket[bkey]["total"] += 1

        task_accuracy = {
            task: counts["correct"] / max(counts["total"], 1)
            for task, counts in by_task.items()
        }
        stratified = {
            bkey: counts["correct"] / max(counts["total"], 1)
            for bkey, counts in by_bucket.items()
        }
        return {
            "overall_accuracy": correct / max(total, 1),
            "task_accuracy": task_accuracy,
            "stratified": stratified,
            "n": total,
        }

    def save_checkpoint(self, epoch: int, metrics: dict):
        """Save model checkpoint and metrics."""
        checkpoint_path = self.output_dir / f"model_epoch_{epoch}.pt"
        torch.save({
            "epoch": epoch,
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "metrics": metrics,
            "config": self.config.to_dict(),
        }, checkpoint_path)

        # Save latest as best_model.pt
        latest_path = self.output_dir / "best_model.pt"
        torch.save({
            "epoch": epoch,
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "metrics": metrics,
            "config": self.config.to_dict(),
        }, latest_path)

    def save_results(self):
        """Save final results."""
        # Save metrics
        metrics_path = self.output_dir / "metrics.json"
        with open(metrics_path, 'w') as f:
            json.dump(self.metrics, f, indent=2)

        # Save config
        config_path = self.output_dir / "config.json"
        with open(config_path, 'w') as f:
            json.dump(self.config.to_dict(), f, indent=2)

    def train(
        self,
        train_loader: DataLoader,
        val_loader: Optional[DataLoader] = None,
        qa_dataset: Optional[List[dict]] = None,
    ):
        """Full training loop.

        Stages:
          STAGE: data    -- dataset / tokenizer / answer-slot coverage stats
          STAGE: token   -- vocab info (size, special-token presence)
          STAGE: init    -- model class & param count vs same-size baseline
          STAGE: train   -- per-epoch: train_loss, val_loss, qa_acc
          STAGE: gen     -- mid-epoch sample generations (visible)
          STAGE: done    -- final summary
        """
        print(f"\n{'='*60}")
        print(f"Experiment: {self.config.exp_id}")
        print(f"Model: {self.config.model}")
        print(f"Device: {self.device}")
        print(f"Parameters: {sum(p.numel() for p in self.model.parameters()):,}")
        print(f"{'='*60}\n")

        # ---- STAGE: data ----
        print(f"STAGE: data exp={self.config.exp_id} "
              f"train_batches={len(train_loader)} val_batches={len(val_loader) if val_loader else 0} "
              f"max_seq_len={self.config.max_seq_len}")
        if qa_dataset:
            from collections import Counter
            ttc = Counter(s["task_type"] for s in qa_dataset if s.get("question"))
            print(f"  qa={len([s for s in qa_dataset if s.get('question')])} per_task={dict(ttc)}")

        # ---- STAGE: token ----
        ans_lit = self.tokenizer.encode("Answer: ", max_len=None)
        print(f"STAGE: token exp={self.config.exp_id} "
              f"vocab_size={self.tokenizer.vocab_size} "
              f"'Answer: ' encodes_to={ans_lit}")

        # ---- STAGE: init ----
        print(f"STAGE: init exp={self.config.exp_id} optimizer=AdamW "
              f"lr={self.config.learning_rate} clip={self.config.gradient_clip}")

        # store the qa snapshot for mid-epoch visibility
        self._qa_snapshot_dataset = list(qa_dataset) if qa_dataset else None

        start_time = time.time()

        for epoch in range(1, self.config.num_epochs + 1):
            epoch_start = time.time()

            # Train (with mid-epoch visibility)
            train_loss, info = self.train_epoch(
                train_loader,
                eval_during_epoch=(qa_dataset is not None and epoch % self.config.eval_every == 0),
            )
            self.metrics["train_loss"].append(train_loss)
            # stash visible mid-epoch snapshots in metrics for offline review
            if info["mid_epoch_samples"]:
                self.metrics.setdefault("mid_epoch_samples", []).extend(
                    [{"epoch": epoch, **s} for s in info["mid_epoch_samples"]])
            if info["mid_epoch_qa"]:
                self.metrics.setdefault("mid_epoch_qa", []).extend(
                    [{"epoch": epoch, **q} for q in info["mid_epoch_qa"]])

            # Validate
            val_loss = None
            if val_loader:
                val_loss = self.evaluate(val_loader)
                self.metrics["val_loss"].append(val_loss)

            # Full QA evaluation
            qa_results = None
            if qa_dataset and epoch % self.config.eval_every == 0:
                qa_results = self.run_qa_eval(qa_dataset)
                self.metrics["eval_accuracy"].append(qa_results)

                # Print + record sample outputs so we can SEE whether the
                # model's output is actually useful, not just low-loss.
                print(f"\n--- [EVAL] exp={self.config.exp_id} epoch {epoch} ---")
                for sample in qa_dataset[:4]:
                    if not sample.get("question"):
                        continue
                    prompt = build_prompt(sample)
                    generated = self.generate(prompt, max_tokens=20, temperature=0.0)
                    predicted = parse_answer(generated).strip().lower()
                    expected = sample["answer"].strip().lower()
                    ok = "✓" if predicted == expected else "✗"
                    print(f"  [{ok}] ({sample['task_type']}) Q: {sample['question']}")
                    print(f"       expected: {sample['answer']!r}  got: {generated!r}")
                    self.metrics["samples"].append({
                        "epoch": epoch,
                        "task_type": sample["task_type"],
                        "prompt": prompt,
                        "expected": sample["answer"],
                        "generated": generated,
                        "correct": predicted == expected,
                    })

            epoch_time = time.time() - epoch_start

            # Log
            log_msg = f"Epoch {epoch}/{self.config.num_epochs} | "
            log_msg += f"train_loss: {train_loss:.4f} | "
            if val_loss is not None:
                log_msg += f"val_loss: {val_loss:.4f} | "
            if qa_results:
                log_msg += f"qa_acc: {qa_results['overall_accuracy']:.3f} | "
            log_msg += f"time: {epoch_time:.1f}s"
            print(log_msg)
            print(f"STAGE: train exp={self.config.exp_id} ep={epoch}/{self.config.num_epochs} "
                  f"train_loss={train_loss:.4f} "
                  f"val_loss={val_loss if val_loss is not None else '-'} "
                  f"qa_acc={qa_results['overall_accuracy'] if qa_results else '-'}")

            # Save checkpoint
            if epoch % self.config.save_every == 0 or epoch == self.config.num_epochs:
                self.save_checkpoint(epoch, {
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "qa_accuracy": qa_results["overall_accuracy"] if qa_results else None,
                })

        total_time = time.time() - start_time
        print(f"\nTraining complete in {total_time:.1f}s")

        # Diagnostic: flag the "low loss but useless output" failure mode.
        if self.metrics["val_loss"] and self.metrics["eval_accuracy"]:
            final_val = self.metrics["val_loss"][-1]
            final_acc = self.metrics["eval_accuracy"][-1]["overall_accuracy"]
            if final_val < 2.0 and final_acc < 0.5:
                print("\n  ⚠ LOW LOSS BUT USELESS OUTPUT")
                print(f"    val_loss={final_val:.3f} (low) yet exact-match acc={final_acc:.3f} (low).")
                print("    The model minimized cross-entropy without learning to answer.")

        # Save final results
        self.save_results()

        # Save samples to text file
        samples_path = self.output_dir / "samples.txt"
        with open(samples_path, 'w') as f:
            for sample in self.metrics["samples"]:
                mark = "CORRECT" if sample.get("correct") else "WRONG"
                f.write(f"\n--- Epoch {sample['epoch']} [{sample['task_type']}] {mark} ---\n")
                f.write(f"Prompt:\n{sample['prompt']}\n")
                f.write(f"Expected:  {sample['expected']}\n")
                f.write(f"Generated: {sample['generated']}\n")

        return self.metrics


In [ ]:
%%writefile src/diagnostics.py
"""
Diagnostics for the hybrid latent-state experiments.

The previous run trained end-to-end but reported ONLY a final 0.0 QA accuracy,
so we could not tell whether failure was due to:
  (a) lack of data diversity,
  (b) a training problem (loss not moving / diverging), or
  (c) a math/architecture bug (I/O path or algebra wrong).

This module produces, on every run, the evidence to separate those three:

  * dataset_diversity()  -> is the data rich enough / biased?
  * io_oracle_tests()    -> does the I/O + answer head work given the TRUE
                            teacher state?  (isolates math from training)
  * make_b_recovery()    -> given the real narrative, how close is make_B(A) to
                            the true answer state B?  (isolates algebra training)
  * token_histogram()    -> what kind of garbage are we generating?
                           (pad spam "."/"..." => decode/I-O broken)
  * render_curves()      -> loss + accuracy-over-time PNG.

All functions are pure-ish (take modules + data, return dicts) so they can be
called from train_modules.py and bench.py.
"""
from collections import Counter
import json
from pathlib import Path

import torch
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# (a) Data diversity
# ---------------------------------------------------------------------------

def dataset_diversity(dataset: list, tokenizer) -> dict:
    """How rich / biased is the data? Prints the 'is it the data?' answer."""
    per_task = Counter(s["task_type"] for s in dataset)
    qa = [s for s in dataset if s.get("question")]
    answers = Counter(s["answer"] for s in qa)
    # majority-class baseline: always predict the most common answer
    majority = answers.most_common(1)[0][1] if answers else 0
    maj_baseline = majority / max(len(qa), 1)

    lens = [len(s["answer"]) for s in qa]
    avg_len = sum(lens) / max(len(lens), 1)

    # recall passwords should be high-entropy / many unique
    recall = [s for s in qa if s["task_type"] == "recall"]
    recall_unique = len(set(s["answer"] for s in recall))

    return {
        "n_total": len(dataset),
        "per_task": dict(per_task),
        "n_qa": len(qa),
        "n_unique_answers": len(answers),
        "most_common_answer": answers.most_common(1)[0] if answers else None,
        "majority_baseline_acc": round(maj_baseline, 4),
        "avg_answer_chars": round(avg_len, 2),
        "recall_n": len(recall),
        "recall_unique_passwords": recall_unique,
        "verdict": (
            "data looks diverse (low majority baseline, many unique answers)"
            if maj_baseline < 0.2 and len(answers) > 10
            else "data may be biased toward a few answers (majority baseline high)"
        ),
    }


# ---------------------------------------------------------------------------
# (b/c) I/O + algebra isolation
# ---------------------------------------------------------------------------

@torch.no_grad()
def io_oracle_tests_with_decoder(encoder, decoder, composer, ans_dec, make_b,
                                 dataset, tokenizer, device, max_new=48) -> dict:
    """Same as io_oracle_tests but with the actual StateDecoder available."""
    eos = tokenizer.vocab[tokenizer.eos_token]
    pad = tokenizer.vocab[tokenizer.pad_token]
    qa = [s for s in dataset if s.get("question")]

    # 1. autoencoder reconstruction char-accuracy (narrative -> narrative)
    recon_hits, recon_tot = 0, 0
    for s in qa[:30]:
        ids = _enc_ids(s["narrative"], tokenizer).unsqueeze(0).to(device)
        states = encoder.states(ids)
        logits = decoder(states)
        pred = logits.argmax(-1)[0]
        tgt = ids[0]
        recon_hits += int((pred == tgt).sum())
        recon_tot += int(tgt.numel())
    autoenc_recon_char_acc = recon_hits / max(recon_tot, 1)

    # 2. oracle answer-head accuracy (teacher state D_target)
    oracle_correct, oracle_tot = 0, 0
    composer_mse_sum, composer_mse_n = 0.0, 0
    for s in qa:
        A_seq = encoder.states(_enc_ids(s["narrative"], tokenizer).unsqueeze(0).to(device)).detach()
        A = A_seq[:, -1, :]
        B = encoder.state_of(_enc_ids(s["answer"], tokenizer).unsqueeze(0).to(device)).detach()
        C = encoder.state_of(_enc_ids(s.get("question", ""), tokenizer).unsqueeze(0).to(device)).detach()
        D_target = encoder.state_of(_enc_ids("Answer: " + s["answer"], tokenizer).unsqueeze(0).to(device)).detach()
        # oracle: decode the teacher target directly
        ids = ans_dec.generate(D_target, max_tokens=max_new, eos_id=eos, pad_id=pad)
        gen = tokenizer.decode(ids).strip().lower()
        exp = s["answer"].strip().lower()
        oracle_correct += int(gen == exp)
        oracle_tot += 1
        # how far does the real pipeline get from D_target?
        D = composer(A, make_b(A_seq, C), C)
        composer_mse_sum += float(F.mse_loss(D, D_target).item())
        composer_mse_n += 1

    return {
        "autoenc_recon_char_acc": round(autoenc_recon_char_acc, 4),
        "oracle_answer_head_acc": round(oracle_correct / max(oracle_tot, 1), 4),
        "oracle_n": oracle_tot,
        "composer_D_mse": round(composer_mse_sum / max(composer_mse_n, 1), 4),
        "interpretation": _interpret_oracle(
            autoenc_recon_char_acc, oracle_correct / max(oracle_tot, 1),
            composer_mse_sum / max(composer_mse_n, 1)),
    }


def _interpret_oracle(autoenc, oracle, composer_mse):
    # The precise I/O test is `oracle` (decode the TRUE teacher state -> answer).
    # autoenc_recon_char_acc (reconstructing a long narrative perfectly) is only
    # informational, since even a good autoencoder rarely hits >0.5 char-acc on
    # long text. We gate on `oracle`, not on autoenc.
    if oracle < 0.5:
        return ("MATH: the answer head cannot decode even the TRUE teacher state "
                f"(<0.5, got {oracle:.2f}) -- the generation head / I-O mapping is wrong.")
    if composer_mse > 0.05:
        return ("TRAINING/ALGEBRA: I/O + head are fine (oracle high) but the composer "
                "does not reach the teacher state (high MSE) -- the latent algebra "
                "(make_B / composer) is not learning.")
    if makeb.get("collapsing"):
        return ("ALGEBRA COLLAPSE: make_B sits at the mean-prediction floor "
                f"(MSE={makeb['make_B_mse']} ~ var={makeb['target_var']}) -- the "
                "single pooled narrative state is too lossy to extract the answer; "
                "use attention over the narrative's token states instead.")
    return ("ALL COMPONENTS OK: I/O, head, and algebra align. Remaining errors are "
            "genuine reasoning difficulty, not a structural bug.")


@torch.no_grad()
def make_b_recovery(encoder, make_b, dataset, tokenizer, device) -> dict:
    """Given the real narrative, how close is make_B(A_seq, C) to the true B?

    Also computes the target's per-sample variance (the MSE floor a mean
    predictor would hit). If achieved MSE ~= variance floor, the module has
    COLLAPSED to the mean -> the input representation lacks the needed info.
    """
    qa = [s for s in dataset if s.get("question")]
    tot, n = 0.0, 0
    Bs = []
    for s in qa:
        A_seq = encoder.states(_enc_ids(s["narrative"], tokenizer).unsqueeze(0).to(device)).detach()
        C = encoder.state_of(_enc_ids(s.get("question", ""), tokenizer).unsqueeze(0).to(device)).detach()
        B = encoder.state_of(_enc_ids(s["answer"], tokenizer).unsqueeze(0).to(device)).detach()
        pred = make_b(A_seq, C)
        tot += float(F.mse_loss(pred, B).item())
        n += 1
        Bs.append(B[0])
    mse = tot / max(n, 1)
    var = float(torch.stack(Bs).var(0).mean().item())   # mean per-dim variance
    collapsing = bool(mse >= 0.9 * var)
    verdict = ("COLLAPSING to mean (MSE ~ variance floor: input state lacks info)"
              if collapsing else ("weak" if mse > 0.08 else "good"))
    return {"make_B_mse": round(mse, 4), "target_var": round(var, 4),
            "collapsing": collapsing, "verdict": verdict}


# ---------------------------------------------------------------------------
# (c) Generation pathology
# ---------------------------------------------------------------------------

def token_histogram(generated_texts: list, tokenizer) -> dict:
    """What are we actually producing? pad spam (".....") => decode broken."""
    pad = tokenizer.pad_token
    eos = tokenizer.eos_token
    unk = tokenizer.unk_token
    counts = Counter()
    for t in generated_texts:
        for ch in t:
            if ch == pad:
                counts["pad"] += 1
            elif ch == eos:
                counts["eos"] += 1
            elif ch == unk:
                counts["unk"] += 1
            elif ch in " \n\t":
                counts["space"] += 1
            elif ch.isalpha():
                counts["alpha"] += 1
            else:
                counts["other"] += 1
    total = sum(counts.values()) or 1
    frac = {k: round(v / total, 3) for k, v in counts.items()}
    if frac.get("pad", 0) > 0.4 or frac.get("other", 0) > 0.4:
        verdict = "GENERATION BROKEN: mostly pad/garbage tokens"
    elif frac.get("alpha", 0) > 0.6:
        verdict = "produces alphabetic text (plausible answers)"
    else:
        verdict = "mixed / unclear output"
    return {"fractions": frac, "verdict": verdict}


# ---------------------------------------------------------------------------
# Plots
# ---------------------------------------------------------------------------

def render_curves(history: dict, out_path: str) -> bool:
    """Plot loss + accuracy over time. Returns True if a PNG was written.

    `history` keys: 'phase0_loss':[...], 'phase1': {name: [mse...]},
    'qa_acc':[...], 'qa_task': {task:[...]}.
    """
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except Exception as e:
        print(f"  (render_curves skipped, matplotlib unavailable: {e})")
        return False

    epochs = list(range(1, len(history.get("qa_acc", [])) + 1))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # left: QA accuracy over time (overall + per task)
    ax = axes[0]
    if history.get("qa_acc"):
        ax.plot(epochs, history["qa_acc"], "k-o", label="overall")
    for task, vals in history.get("qa_task", {}).items():
        if vals:
            ax.plot(range(1, len(vals) + 1), vals, "-", label=task)
    ax.set_title("QA exact-match accuracy over time")
    ax.set_xlabel("eval epoch"); ax.set_ylabel("accuracy")
    ax.set_ylim(-0.05, 1.05); ax.legend(fontsize=7); ax.grid(alpha=0.3)

    # right: phase-1 module MSE over time (training health)
    ax = axes[1]
    for name, vals in history.get("phase1", {}).items():
        if vals:
            ax.plot(range(1, len(vals) + 1), vals, "-", label=name)
    if history.get("phase0_loss"):
        ax.plot(range(1, len(history["phase0_loss"]) + 1),
                history["phase0_loss"], "k--", label="autoenc")
    ax.set_title("Module training loss (MSE) over time")
    ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend(fontsize=7); ax.grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig(out_path, dpi=110)
    plt.close(fig)
    return True


# ---------------------------------------------------------------------------
# helpers
# ---------------------------------------------------------------------------

def _enc_ids(text, tokenizer):
    if not text:
        text = " "
    return torch.tensor([tokenizer.vocab.get(c, tokenizer.vocab[tokenizer.unk_token])
                         for c in text], dtype=torch.long)


def save_telemetry(path: str, obj: dict):
    Path(path).write_text(json.dumps(obj, indent=2))


In [ ]:
%%writefile src/modules.py
"""
Separable, individually-trainable modules for the hybrid latent-state LM.

Design (per project direction): build small input->output functions, each with
its OWN training objective, instead of one monolithic end-to-end model trained
with a single next-token loss. The user's latent-space algebra:

    make_B(A)   -> B                  narrative state A -> answer state B
    make_A(B)   -> A                  inverse
    continue(A) -> A2                 advance narrative state (state evolution)
    continue(B) -> B2                 advance answer state
    Answer_in_format_D(A,B,C) -> D    compose narrative+answer+question -> D

Plus, separately trainable I/O and support modules:
    token -> state   (TokenEncoder)
    state -> token   (StateDecoder)
    state reasoning  (ReasoningStep)
    context management (ContextManager)
    tape prefix embedding / active recall (Tape)

Object mapping for the toy-world tasks:
    A = latent state of the narrative
    B = latent state of the answer
    C = latent state of the question
    D = latent state of the answer rendered in "Answer: <x>" format

Each module is a plain nn.Module with its own forward contract. train_modules.py
trains each one separately (curriculum: first the autoencoder 'output sane
words', then the latent ops), so the latent state is forced to *contain the
correct thing* rather than just learning to spell tokens.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# I/O adapters
# ---------------------------------------------------------------------------

class TokenEncoder(nn.Module):
    """tokens -> state. Returns the final latent state AND all per-token states."""

    def __init__(self, vocab_size: int, d_state: int = 256, d_model: int = 128,
                 n_layers: int = 2):
        super().__init__()
        self.d_state = d_state
        self.embed = nn.Embedding(vocab_size, d_model)
        self.proj = nn.Linear(d_model, d_state)
        # Recurrent core builds the state. LSTM is simple + trainable; the
        # final hidden state is the sequence-level state A/B/C.
        self.core = nn.LSTM(d_state, d_state, num_layers=n_layers, batch_first=True)
        self.norm = nn.LayerNorm(d_state)

    def forward(self, ids: torch.Tensor, return_all: bool = False):
        # ids: [B, T]
        x = self.proj(self.embed(ids))                 # [B, T, d_state]
        out, (h, _) = self.core(x)                     # out: [B, T, d_state]
        final = self.norm(h[-1])                       # [B, d_state]
        return (final, out) if return_all else final

    def states(self, ids: torch.Tensor) -> torch.Tensor:
        """Per-token states [B, T, d_state] (for continue() training)."""
        _, out = self.forward(ids, return_all=True)
        return out

    def state_of(self, ids: torch.Tensor) -> torch.Tensor:
        """Sequence-level final state [B, d_state]."""
        return self.forward(ids)


class StateDecoder(nn.Module):
    """state -> token logits. The 'output sane words' renderer."""

    def __init__(self, d_state: int, vocab_size: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_state, hidden), nn.SiLU(),
            nn.Linear(hidden, vocab_size),
        )

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        # state: [B, d_state] or [B, T, d_state] -> logits same shape minus last
        return self.net(state)


# ---------------------------------------------------------------------------
# Latent-space algebra (all state->state maps)
# ---------------------------------------------------------------------------

class StateTransform(nn.Module):
    """Generic state->state map. Used for make_B, make_A, continue_A, continue_B.

    Small residual MLP so each transform is cheap and independently trainable.
    """

    def __init__(self, d_state: int, hidden: int = 512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_state, hidden), nn.SiLU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, d_state),
        )

    def forward(self, s: torch.Tensor) -> torch.Tensor:
        return self.net(s)


class AnswerComposer(nn.Module):
    """(A, B, C) -> D : compose narrative + answer + question states.

    D_target (the teacher) depends ONLY on the answer (it is
    encoder.state_of("Answer: " + answer)), so B = encoder.state_of(answer) is
    already the answer and must flow into D. We keep a residual from B so the
    answer information can never be lost in the MLP, and widen the net. The
    composer's only job is then to learn the small "Answer: " prefix
    transformation on top of B."""

    def __init__(self, d_state: int, hidden: int = 1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_state * 3, hidden), nn.SiLU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden), nn.SiLU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, d_state),
        )
        # Residual straight from B (the answer state) so D always contains the
        # answer even before the MLP learns the prefix transform.
        self.b_res = nn.Linear(d_state, d_state)

    def forward(self, A: torch.Tensor, B: torch.Tensor, C: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([A, B, C], dim=-1)) + self.b_res(B)


class AnswerDecoder(nn.Module):
    """Autoregressive head: starting from a composed state D, emit the ANSWER
    token sequence (NOT "Answer: ..." -- just the answer string).

    Why not an LSTM seeded from D?  We tried that and the previous Kaggle run
    finished with `oracle_answer_head_acc=0.00` -- meaning even when fed the
    TRUE teacher state, the LSTM could not decode it back to the answer, never
    mind at compositing time. The cause was that generation-time uses a
    zero-valued `start_emb` to roll the LSTM, so the only signal the LSTM sees
    is the initial hidden state h0 = proj(D); this requires the linear->LSTM
    alignment to be learned FROM SCRATCH through a single forward pass, which
    converged to a constant output.

    Instead, this is a NON-recurrent per-position MLP decoder:
      logit_t = MLP([D; pos_embed_t])                  (shape: [B, T, vocab])
    where the MLP is the same per position (weight sharing). It conditions at
    every step on the state D, so even at generation time the prediction has
    rich signal. Teacher-forcing at training time and greedy argmax at
    generation time. Maximum answer length is fixed by MAX_TOKENS to bound the
    parameters; padding past the true end is masked out by the cross-entropy
    (ignore_index=pad).
    """

    MAX_TOKENS = 48  # longest answer in the toy dataset tokenizes to 45; 48 covers it

    def __init__(self, d_state: int, vocab_size: int, hidden: int = 512, max_tokens: int = MAX_TOKENS):
        super().__init__()
        self.d_state = d_state
        self.vocab_size = vocab_size
        self.max_tokens = max_tokens
        # Learnable positional embeddings -> MLP -> logits.
        self.pos_embed = nn.Parameter(torch.randn(max_tokens, d_state) * 0.02)
        self.net = nn.Sequential(
            nn.Linear(d_state + d_state, hidden), nn.SiLU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, vocab_size),
        )
        # Independent projection of D so the decoder isn't forced to read the
        # entire state through one path -- a residual that always has the right
        # output shape, so even at the START of training the state -> vocab
        # linear at least produces a vocab-shaped output (not garbage from
        # breaking activations). This is what the old LSTM head violently
        # lacked -- an anchor path from state -> logit.
        self.state_proj = nn.Linear(d_state, vocab_size)

    def forward_teacher(self, D: torch.Tensor, tgt_ids: torch.Tensor) -> torch.Tensor:
        """Teacher-forced forward -- produces logits for ALL T target positions.

        Args:
            D: [B, d_state] - the composed state to decode.
            tgt_ids: [B, T_raw] - answer token ids (variable length per sample).

        Returns:
            logits: [B, T, vocab] aligned to tgt_ids[:, :T], for cross-entropy,
                    where T = min(T_raw, max_tokens). The target is truncated to
                    T internally so the caller can pass raw (un-padded) answer
                    ids directly without a length mismatch.
        """
        B, T_raw = tgt_ids.shape
        T = min(T_raw, self.max_tokens)
        tgt = tgt_ids[:, :T]                              # [B, T] aligned target
        D_e = D.unsqueeze(1).expand(-1, T, -1)            # [B, T, d]
        pos = self.pos_embed[:T].unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([D_e, pos], dim=-1)                # [B, T, 2d]
        return self.net(h) + self.state_proj(D_e)        # [B, T, vocab]

    @torch.no_grad()
    def generate(self, D: torch.Tensor, max_tokens: int = None,
                 eos_id=None, pad_id=None):
        """Greedy generation: returns a list of token ids (the answer).

        Computes all T positions at once (non-recurrent) and then truncates at
        the FIRST eos (early stop) before stripping any remaining pad/eos. This
        is what makes training-with-EOS work: the model learns to emit EOS
        right after the answer, and `generate` must honor that by cutting the
        tail *before* the strip, not by emitting a full 24-token garbage tail
        and only removing eos/pad at the very end.
        """
        B = D.size(0)
        T = min(max_tokens or self.max_tokens, self.max_tokens)
        D_e = D.unsqueeze(1).expand(B, T, -1)
        pos = self.pos_embed[:T].unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([D_e, pos], dim=-1)
        out = self.net(h) + self.state_proj(D_e)         # [B, T, vocab]
        argmax = out.argmax(-1)
        # The current pipeline only calls this with B==1 (per sample).
        ids = argmax[0].tolist()
        # Early stop at the first EOS (honor the trained stopping signal).
        if eos_id is not None and eos_id in ids:
            ids = ids[:ids.index(eos_id)]
        # Strip any leading/trailing pad (eos already handled by truncation).
        if pad_id is not None:
            ids = [t for t in ids if t != pad_id]
        return ids


# ---------------------------------------------------------------------------
# Support pieces (each separable + trainable on its own)
# ---------------------------------------------------------------------------

class ReasoningStep(nn.Module):
    """State reasoning: apply a transform K times in latent space (the 'think'
    loop). Trained by a consistency/evolution objective in train_modules.py."""

    def __init__(self, d_state: int, steps: int = 4, hidden: int = 512):
        super().__init__()
        self.steps = steps
        self.net = nn.Sequential(
            nn.Linear(d_state, hidden), nn.SiLU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, d_state),
        )

    def forward(self, s: torch.Tensor, steps: int = None) -> torch.Tensor:
        steps = steps or self.steps
        for _ in range(steps):
            s = s + self.net(s)          # residual thinking
        return s


class ContextManager(nn.Module):
    """Context management: soft-pool recent states into a fixed context vector.

    Trained to predict the next state from a window of recent states (so it
    learns 'what is relevant now' instead of attending to the whole stream)."""

    def __init__(self, d_state: int, hidden: int = 256):
        super().__init__()
        self.attn = nn.Linear(d_state, 1)
        self.merge = nn.Sequential(
            nn.Linear(d_state, hidden), nn.SiLU(),
            nn.Linear(hidden, d_state),
        )

    def forward(self, states: torch.Tensor) -> torch.Tensor:
        # states: [B, T, d_state] -> context [B, d_state]
        w = torch.softmax(self.attn(states), dim=1)         # [B, T, 1]
        pooled = (w * states).sum(dim=1)                    # [B, d_state]
        return self.merge(pooled)


class Tape(nn.Module):
    """Prefix tape memory: exact recall. Two separable ops:
      - write(keys, vals): build a key->val memory from a sequence (prefix embed)
      - recall(query): attend over the tape to retrieve (active recall)
    Trained with a retrieval (key->val) objective, separate from reasoning."""

    def __init__(self, d_state: int, hidden: int = 256):
        super().__init__()
        self.k_proj = nn.Linear(d_state, hidden)
        self.v_proj = nn.Linear(d_state, hidden)
        self.q_proj = nn.Linear(d_state, hidden)
        self.out = nn.Linear(hidden, d_state)

    def write(self, states: torch.Tensor):
        # states: [B, T, d_state] -> (keys, vals) stored on the tape
        return self.k_proj(states), self.v_proj(states)

    def recall(self, query: torch.Tensor, tape):
        # query: [B, d_state]; tape = (keys, vals) [B, T, h]
        keys, vals = tape
        q = self.q_proj(query)                              # [B, h]
        scores = torch.einsum("bh,bth->bt", q, keys)        # [B, T]
        w = torch.softmax(scores, dim=1)
        retrieved = torch.einsum("bt,bth->bh", w, vals)     # [B, h]
        return self.out(retrieved)                          # [B, d_state]


# ---------------------------------------------------------------------------
# Compositional inference (ties the pieces together at test time)
# ---------------------------------------------------------------------------

class StateCrossAttn(nn.Module):
    """Extract the answer state B by attending over the narrative's per-token
    states A_seq, conditioned on the question state C.

    Replaces the old make_B(A_vec)->B single-vector MLP, which COLLAPSED to the
    mean (MSE ~ variance of B) because a single pooled narrative vector is too
    lossy to recover a specific fact from a long context. Attention over the
    token-level states is the mechanism that actually lets the model 'read the
    story and pull out the answer'.
    """

    def __init__(self, d_state: int, n_heads: int = 4):
        super().__init__()
        assert d_state % n_heads == 0
        self.d_state = d_state
        self.n_heads = n_heads
        self.q = nn.Linear(d_state, d_state)
        self.k = nn.Linear(d_state, d_state)
        self.v = nn.Linear(d_state, d_state)
        self.out = nn.Linear(d_state, d_state)
        self.norm = nn.LayerNorm(d_state)

    def forward(self, A_seq: torch.Tensor, C: torch.Tensor) -> torch.Tensor:
        # A_seq: [B, T, d]  (per-token narrative states)
        # C:     [B, d]     (question state, used as the attention query)
        B_ = A_seq.size(0)
        dk = self.d_state // self.n_heads
        q = self.q(C).view(B_, self.n_heads, 1, dk)
        k = self.k(A_seq).view(B_, self.n_heads, -1, dk)
        v = self.v(A_seq).view(B_, self.n_heads, -1, dk)
        scores = (q * k).sum(-1) / (dk ** 0.5)          # [B, H, T]
        w = torch.softmax(scores, dim=-1)
        ctx = (w.unsqueeze(-1) * v).sum(-2)              # [B, H, dk]
        ctx = ctx.reshape(B_, self.d_state)
        return self.norm(self.out(ctx))


@torch.no_grad()
def compose_answer(encoder, make_b, composer, ans_dec, A_seq, C, max_tokens=24,
                   eos_id=None, pad_id=None):
    """Run the latent algebra end-to-end to produce an answer string.

    A_seq = narrative per-token states, C = question state.
    B = make_B(A_seq, C)  (attention over the narrative);
    D = composer(A_vec, B, C);  ans_dec generates the answer tokens from D.
    """
    A_vec = A_seq[:, -1, :]                        # [B, d] pooled narrative state
    B = make_b(A_seq, C)
    D = composer(A_vec, B, C)
    ids = ans_dec.generate(D, max_tokens=max_tokens, eos_id=eos_id, pad_id=pad_id)
    return ids


In [ ]:
%%writefile bench.py
#!/usr/bin/env python3
"""
Reusable benchmark for the hybrid latent-state language model.

One entry point for everything:
  * Train one or more model variants.
  * Strict greedy exact-match eval (per task AND by difficulty bucket).
  * Print + save a rich report (loss curves, per-task acc, stratified acc,
    sample outputs, params, "low loss but useless output" flag).

Usage:
  # Fast CPU smoke of every model (tiny):
  python bench.py --quick

  # One model, more serious:
  python bench.py --models baseline --epochs 20 --n_samples 4000 --device cuda

  # Full battery on GPU:
  python bench.py --models baseline,latent_ssm,latent_ssm_decoder --epochs 30 --n_samples 8000 --device cuda

  # Just report existing experiment dirs (e.g. downloaded from Kaggle):
  python bench.py --analyze

The point of the strict, multi-dimensional eval is to catch the failure mode
where cross-entropy loss drops but the model still can't answer (low loss,
useless output). We never report a single number.
"""
import argparse
import json
import os
import random
import sys
import time
from pathlib import Path

import torch

sys.path.insert(0, str(Path(__file__).parent))
from src.dataset import (
    generate_dataset, format_for_training, build_prompt, parse_answer, _bucket,
)
from src.tokenizer import CharTokenizer
from src.models import BaselineTransformer, LatentSSM, LatentSSMDecoder
from src.trainer import Trainer, ExperimentConfig, TextDataset, DataLoader


# Model registry: key -> (model_type, latent_steps, think_every, tokens_per_step, d_model)
# `d_model` here is the per-spec override; latent models also use it as d_state.
# baseline_big is scaled to ~33M params to give a *same-size* autoregressive
# reference for the 34M latent models (the "first night win condition" needs
# an equal-parameter comparison, not 2M baseline vs 34M latent).
MODELS = {
    "baseline":             dict(model="baseline",           ls=0, te=0, tps=8, dm=256),
    "baseline_big":         dict(model="baseline_big",       ls=0, te=0, tps=8, dm=768),
    "latent_ssm":           dict(model="latent_ssm",         ls=0, te=0, tps=8, dm=256),
    "latent_ssm_think":     dict(model="latent_ssm",         ls=4, te=4, tps=8, dm=256),
    "latent_ssm_think8":    dict(model="latent_ssm",         ls=4, te=8, tps=8, dm=256),
    "latent_ssm_decoder":   dict(model="latent_ssm_decoder", ls=4, te=4, tps=8, dm=256),
}

QUICK = dict(d_model=32, n_samples=64, epochs=1, max_seq_len=160,
             location_max_chars=200, inventory_max_chars=200,
             transfer_max_chars=200, recall_max_chars=200, eval_every=1, batch_size=16)

# In --quick mode, force the oversized 'baseline_big' (dm=768 hardcoded in
# MODELS) down to the small d_model so the sanity run actually finishes fast.
QUICK_DM_OVERRIDE = True


def build_model(spec: dict, vocab_size: int, d_model: int):
    m = spec["model"]
    dm = spec.get("dm", d_model)
    if m == "baseline":
        return BaselineTransformer(vocab_size=vocab_size, d_model=dm, num_layers=4, nhead=8)
    if m == "baseline_big":
        return BaselineTransformer(vocab_size=vocab_size, d_model=dm, num_layers=6,
                                   nhead=12, dim_ff=2048)
    if m == "latent_ssm":
        return LatentSSM(vocab_size=vocab_size, d_state=dm, d_model=dm,
                         num_ssm_layers=2, latent_steps=spec["ls"], think_every=spec["te"])
    if m == "latent_ssm_decoder":
        return LatentSSMDecoder(vocab_size=vocab_size, d_state=dm, d_model=dm,
                                num_ssm_layers=2, latent_steps=spec["ls"],
                                tokens_per_step=spec["tps"], think_every=spec["te"])
    raise ValueError(m)


def run_one(key: str, args, dataset, device) -> dict:
    spec = MODELS[key]
    dm = spec.get("dm", args.d_model)
    if getattr(args, "quick", False) and QUICK_DM_OVERRIDE:
        dm = args.d_model
    random.seed(args.seed)
    torch.manual_seed(args.seed)

    print(f"STAGE: start exp={key} model={spec['model']} dm={dm} "
          f"steps={spec['ls']} think_every={spec['te']}")

    train_data = dataset[:int(0.8 * len(dataset))]
    val_data = dataset[int(0.8 * len(dataset)):]

    train_texts = [format_for_training(s) for s in train_data]
    val_texts = [format_for_training(s) for s in val_data]
    tokenizer = CharTokenizer(train_texts, max_vocab=256)

    model = build_model(spec, tokenizer.vocab_size, args.d_model).to(device)
    n_params = sum(p.numel() for p in model.parameters())

    config = ExperimentConfig(
        exp_id=key, model=spec["model"], d_model=dm,
        latent_steps=spec["ls"], batch_size=args.batch_size,
        num_epochs=args.epochs, max_seq_len=args.max_seq_len,
        eval_every=args.eval_every, seed=args.seed, device=str(device),
        # Visibility: mid-epoch loss numbers + a fresh generation sample so the
        # user can actually watch the model learn (instead of staring at a
        # blank console for an hour and then a single accuracy=0 result).
        print_every_batches=args.print_every_batches,
        gen_sample_every=args.gen_sample_every,
        # Answer-slot focus: up-weight loss on the "Answer:" continuation so
        # the model gets sharp signal on the *answer* chars, not just filler.
        # 0 disables. 1.0 doubles the loss weight on answer positions.
        answer_loss_weight=args.answer_loss_weight,
    )
    trainer = Trainer(model, config, tokenizer, output_dir=args.output_dir)
    train_loader = DataLoader(TextDataset(train_texts, tokenizer, max_len=args.max_seq_len),
                              batch_size=args.batch_size, shuffle=True)
    val_loader = DataLoader(TextDataset(val_texts, tokenizer, max_len=args.max_seq_len),
                            batch_size=args.batch_size)

    t0 = time.time()
    metrics = trainer.train(train_loader, val_loader, qa_dataset=val_data)
    elapsed = time.time() - t0

    qa = metrics["eval_accuracy"][-1] if metrics["eval_accuracy"] else None
    final_val = metrics["val_loss"][-1] if metrics["val_loss"] else None
    low_but_useless = bool(final_val is not None and final_val < 2.0
                           and qa is not None and qa["overall_accuracy"] < 0.5)

    # A few visible sample outputs (expected vs generated).
    samples = []
    for s in val_data[:6]:
        if not s.get("question"):
            continue
        prompt = build_prompt(s)
        gen = trainer.generate(prompt, max_tokens=20, temperature=0.0)
        samples.append(dict(task=s["task_type"], question=s["question"],
                            expected=s["answer"], generated=gen.strip()))

    return dict(
        key=key, model=spec["model"], latent_steps=spec["ls"], think_every=spec["te"],
        params=n_params, elapsed_s=elapsed,
        final_train_loss=metrics["train_loss"][-1] if metrics["train_loss"] else None,
        final_val_loss=final_val,
        overall_accuracy=qa["overall_accuracy"] if qa else None,
        task_accuracy=qa["task_accuracy"] if qa else None,
        stratified=qa["stratified"] if qa else None,
        n_eval=qa["n"] if qa else None,
        low_but_useless=low_but_useless,
        samples=samples,
    )
    print(f"STAGE: done exp={key} val_loss={final_val} acc={qa['overall_accuracy'] if qa else None}")


def render_report(results: list) -> str:
    _f = lambda x: f"{x:.3f}" if isinstance(x, (int, float)) else "—"
    lines = []
    lines.append("# Benchmark Report\n")
    lines.append(f"Models evaluated: {len(results)}\n")

    # Summary table
    lines.append("## Summary\n")
    lines.append("| model | steps | params | train_loss | val_loss | "
                 "exact_acc | low_loss_useless |")
    lines.append("|---|---|---|---|---|---|---|")
    for r in results:
        params = f"{r['params']:,}" if isinstance(r["params"], int) else str(r["params"])
        lines.append(
            f"| {r['key']} | {r['latent_steps']} | {params} | "
            f"{_f(r['final_train_loss'])} | {_f(r['final_val_loss'])} | "
            f"{_f(r['overall_accuracy'])} | {'YES' if r['low_but_useless'] else 'no'} |"
        )

    # Per-task accuracy
    lines.append("\n## Per-task exact-match accuracy\n")
    tasks = ["location", "inventory", "transfer", "recall"]
    lines.append("| model | " + " | ".join(tasks) + " |")
    lines.append("|---|" + "|".join(["---"] * len(tasks)) + "|")
    for r in results:
        ta = r["task_accuracy"] or {}
        lines.append("| " + r["key"] + " | " +
                     " | ".join(_f(ta.get(t)) for t in tasks) + " |")

    # Stratified
    lines.append("\n## Stratified by difficulty bucket\n")
    for r in results:
        if not r["stratified"]:
            continue
        lines.append(f"**{r['key']}**")
        for bkey, acc in sorted(r["stratified"].items()):
            lines.append(f"  - {bkey:<26} {_f(acc)}")
        lines.append("")

    # Sample outputs
    lines.append("\n## Sample outputs (expected vs generated)\n")
    for r in results:
        lines.append(f"**{r['key']}** (exact_acc={_f(r['overall_accuracy'])})")
        for s in r["samples"]:
            lines.append(f"  - [{s['task']}] Q: {s['question']}")
            lines.append(f"      expected: {s['expected']!r}")
            lines.append(f"      generated: {s['generated']!r}")
        lines.append("")

    return "\n".join(lines)


def analyze_existing(output_dir: str = "experiments") -> list:
    """Build a report from existing experiment dirs (no training)."""
    results = []
    for d in sorted(Path(output_dir).iterdir()):
        mfile = d / "metrics.json"
        if not mfile.exists():
            continue
        with open(mfile) as f:
            m = json.load(f)
        qa = m["eval_accuracy"][-1] if m.get("eval_accuracy") else None
        final_val = m["val_loss"][-1] if m.get("val_loss") else None
        samples = []
        stxt = d / "samples.txt"
        if stxt.exists():
            txt = stxt.read_text()
            for blk in txt.split("--- Epoch")[1:]:
                for line in blk.splitlines():
                    if line.strip().startswith("Expected:"):
                        samples.append(line.strip())
        results.append(dict(
            key=d.name, model=m.get("config", {}).get("model", "?"),
            latent_steps=m.get("config", {}).get("latent_steps", "?"),
            params="?", elapsed_s=0,
            final_train_loss=m["train_loss"][-1] if m.get("train_loss") else None,
            final_val_loss=final_val,
            overall_accuracy=qa["overall_accuracy"] if qa else None,
            task_accuracy=qa["task_accuracy"] if qa else None,
            stratified=qa["stratified"] if qa else None,
            n_eval=qa["n"] if qa else None,
            low_but_useless=bool(final_val is not None and final_val < 2.0
                                 and qa is not None and qa["overall_accuracy"] < 0.5),
            samples=[dict(task="", question="", expected=s, generated="") for s in samples[:6]],
        ))
    return results


def main():
    ap = argparse.ArgumentParser(description="Reusable latent-state benchmark")
    ap.add_argument("--models", default="baseline,baseline_big,latent_ssm,latent_ssm_think,latent_ssm_decoder",
                    help="Comma-separated model keys from the registry")
    ap.add_argument("--quick", action="store_true", help="Tiny CPU defaults")
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--n_samples", type=int, default=5000)
    ap.add_argument("--d_model", type=int, default=256)
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--max_seq_len", type=int, default=768)
    ap.add_argument("--location_max_chars", type=int, default=600)
    ap.add_argument("--inventory_max_chars", type=int, default=600)
    ap.add_argument("--transfer_max_chars", type=int, default=600)
    ap.add_argument("--recall_max_chars", type=int, default=600)
    ap.add_argument("--eval_every", type=int, default=1)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--output_dir", default="experiments")
    ap.add_argument("--analyze", action="store_true", help="Report existing dirs only")
    ap.add_argument("--print_every_batches", type=int, default=50,
                    help="Visible [TRAIN] log every N batches. 0 to disable.")
    ap.add_argument("--gen_sample_every", type=int, default=200,
                    help="Visible [GEN] sample every N batches mid-epoch. 0 to disable.")
    ap.add_argument("--answer_loss_weight", type=float, default=1.0,
                    help="Extra loss weight on answer-slot tokens (focused loss)."
                         " 0 = standard uniform CE; 1.0 = doubles answer-slot loss.")
    args = ap.parse_args()

    if args.analyze:
        results = analyze_existing(args.output_dir)
        report = render_report(results)
        print(report)
        Path("bench_report.md").write_text(report)
        print(f"\nSaved bench_report.md ({len(results)} experiments)")
        return

    if args.quick:
        for k, v in QUICK.items():
            setattr(args, k, v)

    device = torch.device("cuda" if (args.device == "auto" and torch.cuda.is_available())
                          else ("cpu" if args.device == "auto" else args.device))

    keys = [k.strip() for k in args.models.split(",") if k.strip()]
    print(f"Device: {device} | models: {keys}\n")
    print(f"STAGE: bench start models={keys} epochs={args.epochs} "
          f"n_samples={args.n_samples} device={device}")

    dataset = generate_dataset(
        n_samples=args.n_samples, seed=args.seed,
        location_max_chars=args.location_max_chars,
        inventory_max_chars=args.inventory_max_chars,
        transfer_max_chars=args.transfer_max_chars,
        recall_max_chars=args.recall_max_chars,
    )

    results = []
    for key in keys:
        if key not in MODELS:
            print(f"Unknown model key: {key} (known: {list(MODELS)})")
            continue
        print(f"\n##### {key} #####")
        try:
            results.append(run_one(key, args, dataset, device))
        except Exception as e:
            print(f"FAILED {key}: {e}")

    report = render_report(results)
    print("\n" + report)
    Path("bench_report.md").write_text(report)
    Path("bench_results.json").write_text(json.dumps(results, indent=2))
    print(f"\nSaved bench_report.md and bench_results.json ({len(results)} models)")
    print("STAGE: done")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train_modules.py
#!/usr/bin/env python3
"""
Train the separable latent-state modules, each with its OWN objective.

Curriculum (per project direction):
  Phase 0  token<->state autoencoder  -> "output sane words"
  Phase 1  latent algebra, each piece separate:
            make_B(A)->B, make_A(B)->A, continue(A)->A2, continue(B)->B2,
            Answer_in_format_D(A,B,C)->D  (composed state then decoded)

Each module is trained independently (own optimizer + own loss). The 'remember
the correct thing' objective is make_B(A)->B, where the TARGET B is the encoder
state of the true answer -- so the narrative state is forced to contain the
answer, not just learn to spell tokens.

FEEDBACK / DIAGNOSTICS (added): this script now records a full time series
(per-epoch autoencoder loss, per-module MSE, per-epoch compositional QA accuracy
overall + per task) and runs isolation tests that separate the three possible
failure modes:
  (a) data diversity   -> dataset_diversity()
  (b) training health  -> loss curves over time + composer reaching D_target
  (c) math / I-O       -> io_oracle_tests (decode the TRUE teacher state) and
                          make_b_recovery()
Generated outputs are histogrammed so we can see pad/garbage spam. Results go
to telemetry.json + diagnostics.txt + curves.png (+ modules_report.json for
backwards compatibility with kaggle_ctl.py).

Usage:
  python train_modules.py --quick                      # tiny CPU sanity
  python train_modules.py --device cuda --epochs 25     # real Kaggle run
  python train_modules.py --analyze                     # report last run
"""
import argparse
import json
import math
import os
import sys
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, str(Path(__file__).parent))
from src.dataset import generate_dataset, format_for_training, build_prompt, parse_answer
from src.tokenizer import CharTokenizer
from src.modules import (
    TokenEncoder, StateDecoder, StateTransform, AnswerComposer, AnswerDecoder,
    StateCrossAttn, compose_answer,
)
from src import diagnostics as diag

QUICK = dict(d_state=48, n_samples=400, epochs=4, max_seq_len=192,
             phase0_epochs=8, phase1_epochs=3, d_model=32, batch_size=32,
             max_cycles=2, cycle_epochs=1, plateau_eps=0.05, plateau_patience=2)


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        ids = self.tokenizer.encode(self.texts[i], max_len=self.max_len)
        ids = ids[: self.max_len]
        while len(ids) < self.max_len:
            ids.append(self.tokenizer.vocab[self.tokenizer.pad_token])
        x = torch.tensor(ids, dtype=torch.long)
        return x, x  # reconstruct input from itself


def make_loaders(texts, tokenizer, batch_size, max_len):
    ds = TextDataset(texts, tokenizer, max_len)
    n = len(ds)
    train = TextDataset(texts[: int(0.8 * n)], tokenizer, max_len)
    val = TextDataset(texts[int(0.8 * n):], tokenizer, max_len)
    return (DataLoader(train, batch_size=batch_size, shuffle=True),
            DataLoader(val, batch_size=batch_size))


# ---------------------------------------------------------------------------
# Phase 0: autoencoder (output sane words)
# ---------------------------------------------------------------------------

def train_autoencoder(encoder, decoder, loader, opt, device, epochs, hist, threshold=1.2):
    encoder.train(); decoder.train()
    last = 1e9
    for ep in range(1, epochs + 1):
        tot = 0; n = 0
        for x, _ in loader:
            x = x.to(device)
            opt.zero_grad()
            states = encoder.states(x)              # [B,T,d_state]
            logits = decoder(states)               # [B,T,vocab]
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                   x.reshape(-1), ignore_index=0)
            loss.backward(); opt.step()
            tot += loss.item(); n += 1
        last = tot / max(n, 1)
        hist["phase0_loss"].append(last)
        print(f"  [autoenc] ep {ep}/{epochs} recon_loss={last:.4f}")
        print(f"  STAGE: autoenc ep={ep}/{epochs} loss={last:.4f}")
        if last < threshold:
            print(f"  autoenc below threshold {threshold}; proceeding to latent ops.")
            break
    return last


# ---------------------------------------------------------------------------
# Phase 1: latent algebra (each piece separate)
# ---------------------------------------------------------------------------

def train_make_b_attn(module, samples, encoder, opt, device, epochs, hist, name="make_B(A)->B"):
    """Train make_B as CROSS-ATTENTION over the narrative's per-token states,
    conditioned on the question. (Replaces the old single-vector MLP that
    collapsed to the mean.)"""
    module.train(); encoder.eval()
    crit = nn.MSELoss()
    for ep in range(1, epochs + 1):
        tot = 0; n = 0
        for s in samples:
            A_seq = encoder.states(_ids(s["narrative"]).unsqueeze(0).to(device)).detach()
            C = encoder.state_of(_ids(s.get("question", "")).unsqueeze(0).to(device)).detach()
            B = encoder.state_of(_ids(s["answer"]).unsqueeze(0).to(device)).detach()
            opt.zero_grad()
            pred = module(A_seq, C)
            loss = crit(pred, B)
            loss.backward(); opt.step()
            tot += loss.item(); n += 1
        last = tot / max(n, 1)
        hist["phase1"].setdefault(name, []).append(last)
        print(f"  [{name}] ep {ep}/{epochs} mse={last:.4f}")
        print(f"  STAGE: {name} ep={ep}/{epochs} mse={last:.4f}")
    return last


def train_state_map(module, src_fn, tgt_fn, samples, encoder, opt, device, epochs, name, hist):
    """Train a state->state map with an MSE objective against teacher states.
    (Used for make_A(B)->A; make_B now uses the cross-attention version above.)"""
    module.train(); encoder.eval()
    crit = nn.MSELoss()
    for ep in range(1, epochs + 1):
        tot = 0; n = 0
        for s in samples:
            A = encoder.state_of(_ids(s["narrative"]).unsqueeze(0).to(device)).detach()
            B = encoder.state_of(_ids(s["answer"]).unsqueeze(0).to(device)).detach()
            C = encoder.state_of(_ids(s.get("question", "")).unsqueeze(0).to(device)).detach()
            src = src_fn(A, B, C)
            tgt = tgt_fn(A, B, C)
            opt.zero_grad()
            pred = module(src)
            loss = crit(pred, tgt)
            loss.backward(); opt.step()
            tot += loss.item(); n += 1
        last = tot / max(n, 1)
        hist["phase1"].setdefault(name, []).append(last)
        print(f"  [{name}] ep {ep}/{epochs} mse={last:.4f}")
        print(f"  STAGE: {name} ep={ep}/{epochs} mse={last:.4f}")
    return last


def train_composer(composer, ans_dec, dec, make_b, samples, encoder, opt, device, epochs, hist, qa_hook=None):
    composer.train(); ans_dec.train(); encoder.eval(); make_b.eval()
    for ep in range(1, epochs + 1):
        tot = 0; n = 0
        for s in samples:
            A_seq = encoder.states(_ids(s["narrative"]).unsqueeze(0).to(device)).detach()
            A_vec = A_seq[:, -1, :]                                   # pooled narrative state
            C = encoder.state_of(_ids(s.get("question", "")).unsqueeze(0).to(device)).detach()
            # INFERENCE-PATH B: at test time the answer state is produced by
            # make_b(A_seq, C), NOT the true answer state. Training the composer
            # on the true answer state (encoder.state_of(answer)) made it look
            # perfect (answerD mse 0.0015) while the REAL pipeline fed
            # make_b's output, which is ~0.37 from the true state -- so the
            # composed D was garbage at inference and final QA collapsed to 0.
            # Closing the train/inference gap: feed the exact path the pipeline
            # runs. B is detached so the composer's gradient does not leak into
            # make_b (make_b is trained separately in the toB phase).
            B = make_b(A_seq, C).detach()
            D_target = encoder.state_of(_ids("Answer: " + s["answer"]).unsqueeze(0).to(device)).detach()
            opt.zero_grad()
            D = composer(A_vec, B, C)
            # REPRESENTATION-ONLY objective: latent consistency. The composed
            # state D must equal the answer's OWN state (D_target). We do NOT
            # optimize for the answer string (that would be next-token
            # shortcutting -- the trap the whole project avoids). The
            # AnswerDecoder is a SEPARATE readout PROBE (see train_answer_probe),
            # never part of this loss. The core model is optimized purely for
            # the internal representation; the answer is only *read out* later.
            loss = F.mse_loss(D, D_target)
            loss.backward(); opt.step()
            tot += loss.item(); n += 1
        last = tot / max(n, 1)
        hist["phase1"].setdefault("composer", []).append(last)
        print(f"  [composer] ep {ep}/{epochs} loss={last:.4f}")
        print(f"  STAGE: composer ep={ep}/{epochs} loss={last:.4f}")
        if qa_hook is not None and (ep % 3 == 0 or ep == epochs):
            qa_hook(ep)
    return last


def train_answer_probe(ans_dec, composer, samples, encoder, opt, device, epochs):
    """Readout PROBE (NOT a core loss). Train the AnswerDecoder to read the
    answer from the teacher state D_target, with Gaussian noise injected at the
    composer's CURRENT error level so the probe is robust to exactly the noise
    it will face at inference (composer(A,B,C) ~ D_target + that noise).

    This is deliberately decoupled from the core model: it never puts an answer
    gradient into the composer / transforms / autoencoder. We optimize the
    INTERNAL REPRESENTATION elsewhere; this head only learns to *read it out*.
    """
    ans_dec.train(); encoder.eval(); composer.eval()
    eos = TOK.vocab[TOK.eos_token]
    crit = nn.CrossEntropyLoss()
    for ep in range(1, epochs + 1):
        tot = 0; n = 0
        for s in samples:
            A = encoder.state_of(_ids(s["narrative"]).unsqueeze(0).to(device)).detach()
            B = encoder.state_of(_ids(s["answer"]).unsqueeze(0).to(device)).detach()
            C = encoder.state_of(_ids(s.get("question", "")).unsqueeze(0).to(device)).detach()
            D_target = encoder.state_of(_ids("Answer: " + s["answer"]).unsqueeze(0).to(device)).detach()
            D = composer(A, B, C).detach()
            with torch.no_grad():
                cur_mse = F.mse_loss(D, D_target).item()
                noise_std = math.sqrt(cur_mse) + 1e-4
                D_train = D_target + torch.randn_like(D_target) * noise_std
            ans_ids = _ids(s["answer"]).to(device)
            tgt = torch.cat([ans_ids, torch.tensor([eos], device=device)])
            opt.zero_grad()
            logits = ans_dec.forward_teacher(D_train, tgt.unsqueeze(0))
            loss = crit(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))
            loss.backward(); opt.step()
            tot += loss.item(); n += 1
        last = tot / max(n, 1)
        print(f"  [answer_probe] ep {ep}/{epochs} ce={last:.4f}")
    return last


def dump_validation_examples(encoder, make_b, composer, ans_dec, dataset,
                             tokenizer, device, n_per_task=4, max_new=48):
    """Print expected vs generated (pipeline) vs oracle (clean state) for a few
    validation samples per task, so we can CONCLUDE why it worked/failed from
    real examples rather than only aggregate accuracy."""
    eos = tokenizer.vocab[tokenizer.eos_token]
    pad = tokenizer.vocab[tokenizer.pad_token]
    rows = []
    seen = {}
    for s in dataset:
        if not s.get("question"):
            continue
        t = s["task_type"]
        if seen.get(t, 0) >= n_per_task:
            continue
        seen[t] = seen.get(t, 0) + 1
        A_seq = encoder.states(_ids(s["narrative"]).unsqueeze(0).to(device)).detach()
        C = encoder.state_of(_ids(s.get("question", "")).unsqueeze(0).to(device)).detach()
        ids = compose_answer(encoder, make_b, composer, ans_dec, A_seq, C,
                              max_tokens=max_new, eos_id=eos, pad_id=pad)
        gen = tokenizer.decode(ids).strip().lower()
        exp = s["answer"].strip().lower()
        D_target = encoder.state_of(_ids("Answer: " + s["answer"]).unsqueeze(0).to(device)).detach()
        oids = ans_dec.generate(D_target, max_tokens=max_new, eos_id=eos, pad_id=pad)
        ogen = tokenizer.decode(oids).strip().lower()
        rows.append((t, s.get("question", ""), exp, gen, ogen))
    print("\n=== VALIDATION EXAMPLES (expected | pipeline-gen | oracle-gen) ===")
    for t, q, exp, gen, ogen in rows:
        tag = "OK" if gen == exp else ("oracleOK" if ogen == exp else "X")
        print(f"  [{t}] Q: {q}")
        print(f"       exp={exp!r}  gen={gen!r}  oracle={ogen!r}  [{tag}]")
    return rows


# continue(): train on consecutive encoder states. NOTE: optimizer must be
# created ONCE outside the sample loop (the old code re-created it per sample,
# which reset AdamW state every step and crippled learning).
def train_cont(module, samples, encoder, device, epochs, name, field, hist):
    module.train(); encoder.eval()
    crit = nn.MSELoss()
    opt = torch.optim.AdamW(module.parameters(), lr=3e-4)
    for ep in range(1, epochs + 1):
        tot = 0; n = 0
        for s in samples:
            ids = _ids(s[field])
            if ids.size(0) < 2:
                continue
            states = encoder.states(ids.unsqueeze(0).to(device)).detach()[0]  # [T, d]
            src = states[:-1]; tgt = states[1:]
            opt.zero_grad()
            pred = module(src)
            loss = crit(pred, tgt)
            loss.backward(); opt.step()
            tot += loss.item(); n += 1
        last = tot / max(n, 1)
        hist["phase1"].setdefault(name, []).append(last)
        print(f"  [{name}] ep {ep}/{epochs} mse={last:.4f}")
        print(f"  STAGE: {name} ep={ep}/{epochs} mse={last:.4f}")
    return last


# ---------------------------------------------------------------------------
# Eval (compositional inference)
# ---------------------------------------------------------------------------

@torch.no_grad()
def run_qa(encoder, make_b, composer, ans_dec, dataset, tokenizer, device, max_new=48):
    correct = 0; total = 0
    by_task = {}
    generated_texts = []
    eos = tokenizer.vocab[tokenizer.eos_token]
    pad = tokenizer.vocab[tokenizer.pad_token]
    for s in dataset:
        if not s.get("question"):
            continue
        total += 1
        t = s["task_type"]
        by_task.setdefault(t, [0, 0])
        A_seq = encoder.states(_ids(s["narrative"]).unsqueeze(0).to(device)).detach()
        C = encoder.state_of(_ids(s.get("question", "")).unsqueeze(0).to(device)).detach()
        ids = compose_answer(encoder, make_b, composer, ans_dec, A_seq, C,
                             max_tokens=max_new, eos_id=eos, pad_id=pad)
        gen = tokenizer.decode(ids).strip().lower()
        generated_texts.append(gen)
        exp = s["answer"].strip().lower()
        ok = gen == exp
        correct += ok
        by_task[t][0] += ok; by_task[t][1] += 1
    acc = correct / max(total, 1)
    task_acc = {t: c / m for t, (c, m) in by_task.items()}
    return acc, task_acc, total, generated_texts


def _ids(text):
    # encode lazily; tokenizer is set globally. Guard empty text (story tasks
    # have no question) so the LSTM never sees a length-0 sequence.
    if not text:
        text = " "
    return torch.tensor([TOK.vocab.get(c, TOK.vocab[TOK.unk_token]) for c in text],
                        dtype=torch.long)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

TOK = None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--quick", action="store_true")
    ap.add_argument("--device", default="auto")
    ap.add_argument("--epochs", type=int, default=25)
    ap.add_argument("--phase0_epochs", type=int, default=15)
    ap.add_argument("--phase1_epochs", type=int, default=25)
    # Cyclic curriculum: one CYCLE = (toA -> toB -> contA -> contB ->
    # answerWithFormatD). Repeated until ALL step-losses in a cycle plateau.
    ap.add_argument("--max_cycles", type=int, default=25)
    ap.add_argument("--cycle_epochs", type=int, default=3,
                    help="epochs per phase per cycle")
    ap.add_argument("--plateau_eps", type=float, default=0.01,
                    help="max per-step loss change across cycles to count as plateau")
    ap.add_argument("--plateau_patience", type=int, default=3,
                    help="consecutive plateaued cycles before early-stop")
    ap.add_argument("--n_samples", type=int, default=5000)
    ap.add_argument("--d_state", type=int, default=256)
    ap.add_argument("--d_model", type=int, default=128)
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--max_seq_len", type=int, default=512)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--fresh", action="store_true",
                   help="Retrain the autoencoder foundation even if cached")
    ap.add_argument("--analyze", action="store_true", help="Report last run only")
    args = ap.parse_args()

    if args.analyze:
        report_analyze()
        return

    if args.quick:
        for k, v in QUICK.items():
            setattr(args, k, v)

    global TOK
    device = torch.device("cuda" if (args.device == "auto" and torch.cuda.is_available())
                          else ("cpu" if args.device == "auto" else args.device))
    print(f"STAGE: train_modules device={device} d_state={args.d_state} n={args.n_samples}")

    dataset = generate_dataset(n_samples=args.n_samples, seed=args.seed)
    texts = [format_for_training(s) for s in dataset]
    TOK = CharTokenizer(texts, max_vocab=256)

    # ----- (a) DATA DIVERSITY CHECK (run first: is it the data?) -----
    div = diag.dataset_diversity(dataset, TOK)
    print("STAGE: data_diversity " + json.dumps(div))
    print(f"  >> data: {div['n_qa']} QA samples, {div['n_unique_answers']} unique "
          f"answers, majority-baseline acc={div['majority_baseline_acc']} -> {div['verdict']}")

    encoder = TokenEncoder(TOK.vocab_size, d_state=args.d_state, d_model=args.d_model).to(device)
    decoder = StateDecoder(args.d_state, TOK.vocab_size).to(device)
    make_b = StateCrossAttn(args.d_state).to(device)
    make_a = StateTransform(args.d_state).to(device)
    cont_a = StateTransform(args.d_state).to(device)
    cont_b = StateTransform(args.d_state).to(device)
    composer = AnswerComposer(args.d_state).to(device)
    ans_dec = AnswerDecoder(args.d_state, TOK.vocab_size).to(device)

    train_loader, val_loader = make_loaders(texts, TOK, args.batch_size, args.max_seq_len)
    hist = {"phase0_loss": [], "phase1": {}, "qa_acc": [], "qa_task": {}}

    # ---- Phase 0: output sane words (the 'baseline' foundation) ----
    # --quick uses its OWN cache so it never clobbers the real 256-dim
    # foundation; a d_state mismatch triggers a retrain instead of a crash.
    found_ckpt = Path("modules_foundation_quick.pt" if args.quick else "modules_foundation.pt")
    loaded = False
    if found_ckpt.exists() and not args.fresh:
        ck = torch.load(found_ckpt, map_location=device, weights_only=True)
        if ck["encoder"]["norm.weight"].shape[0] == args.d_state:
            print("STAGE: phase0 load cached foundation (skip retrain)")
            encoder.load_state_dict(ck["encoder"])
            decoder.load_state_dict(ck["decoder"])
            loaded = True
        else:
            print(f"STAGE: phase0 cached foundation d_state mismatch "
                  f"({ck['encoder']['norm.weight'].shape[0]} != {args.d_state}); retraining")
    if not loaded:
        print("STAGE: phase0 autoencoder (train)")
        ae_opt = torch.optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=3e-4)
        train_autoencoder(encoder, decoder, train_loader, ae_opt, device, args.phase0_epochs, hist)
        torch.save({"encoder": encoder.state_dict(), "decoder": decoder.state_dict()},
                   found_ckpt)
        print(f"  saved foundation -> {found_ckpt}")

    # sanity: reconstruct a validation sample
    encoder.eval(); decoder.eval()
    with torch.no_grad():
        x = val_loader.dataset[0][0].unsqueeze(0).to(device)
        rec = decoder(encoder.states(x)).argmax(-1)[0]
        print("  sample reconstruction:", TOK.decode(rec.tolist())[:80])

    # ---- Phase 1: latent algebra, CYCLIC curriculum ----
    # ONE CYCLE = (toA -> toB -> contA -> contB -> answerWithFormatD). The
    # regiment keeps conversion (toB), continuation (contA/contB) and inference
    # (answerWithFormatD) balanced and optimizes the INTERNAL REPRESENTATION --
    # NOT the answer string. The AnswerDecoder is a readout PROBE (trained
    # separately, never part of the core loss). Repeat cycles until ALL
    # step-losses plateau (change < plateau_eps across consecutive cycles).
    print("STAGE: phase1 cyclic latent algebra")
    qa_samples = [s for s in dataset[int(0.8 * len(dataset)):] if s.get("question")]

    # QA hook: evaluated periodically during composer training so we get a
    # *learning curve* for the answer, not just a final number.
    def qa_hook(ep):
        acc, task_acc, n, _ = run_qa(encoder, make_b, composer, ans_dec,
                                     qa_samples, TOK, device)
        hist["qa_acc"].append(acc)
        for t, a in task_acc.items():
            hist["qa_task"].setdefault(t, []).append(a)
        print(f"  [qa] ep={ep} acc={acc:.3f} " +
              " ".join(f"{t}={a:.2f}" for t, a in task_acc.items()))
        print(f"  STAGE: qa ep={ep} acc={acc:.3f} " +
              " ".join(f"{t}={a:.2f}" for t, a in task_acc.items()))

    # Persisted optimizers -- one shared model, refined every cycle.
    ae_opt = torch.optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=3e-4)
    makeb_opt = torch.optim.AdamW(make_b.parameters(), lr=3e-4)
    makea_opt = torch.optim.AdamW(make_a.parameters(), lr=3e-4)
    cont_a_opt = torch.optim.AdamW(cont_a.parameters(), lr=3e-4)
    cont_b_opt = torch.optim.AdamW(cont_b.parameters(), lr=3e-4)
    comp_opt = torch.optim.AdamW(composer.parameters(), lr=3e-4)
    probe_opt = torch.optim.AdamW(ans_dec.parameters(), lr=3e-4)

    cycle_losses = []   # list of dicts: {step: loss}
    patience = 0
    for cyc in range(1, args.max_cycles + 1):
        print(f"\n##### CYCLE {cyc}/{args.max_cycles} #####")
        # toA: nudge the shared state space (no-op once foundation < threshold)
        ae_loss = train_autoencoder(encoder, decoder, train_loader, ae_opt,
                                    device, args.cycle_epochs, hist)
        # toB: answer state from narrative + question (conversion)
        makeb_loss = train_make_b_attn(make_b, qa_samples, encoder, makeb_opt,
                                       device, args.cycle_epochs, hist)
        # make_A(B)->A
        makea_loss = train_state_map(make_a, lambda A, B, C: B, lambda A, B, C: A,
                                     qa_samples, encoder, makea_opt, device,
                                     args.cycle_epochs, "make_A(B)->A", hist)
        # contA: continue narrative state
        cont_a_loss = train_cont(cont_a, qa_samples, encoder, device,
                                 args.cycle_epochs, "continue(A)->A2", "narrative", hist)
        # contB: continue answer state
        cont_b_loss = train_cont(cont_b, qa_samples, encoder, device,
                                 args.cycle_epochs, "continue(B)->B2", "answer", hist)
        # answerWithFormatD: compose D (latent consistency only, on the
        # inference path B = make_b(A_seq, C))
        comp_loss = train_composer(composer, ans_dec, decoder, make_b, qa_samples, encoder,
                                   comp_opt, device, args.cycle_epochs, hist, qa_hook=qa_hook)
        # probe: readout head trained on clean teacher state (validation only)
        probe_loss = train_answer_probe(ans_dec, composer, qa_samples, encoder,
                                        probe_opt, device, args.cycle_epochs)
        cyc_loss = {"toA": ae_loss, "toB": makeb_loss, "makeA": makea_loss,
                    "contA": cont_a_loss, "contB": cont_b_loss,
                    "answerD": comp_loss, "probe": probe_loss}
        cycle_losses.append(cyc_loss)
        print("STAGE: cycle " + " ".join(f"{k}={v:.4f}" for k, v in cyc_loss.items()))
        # Plateau: stop when ALL step-losses change < eps vs the previous cycle.
        if len(cycle_losses) >= 2:
            prev = cycle_losses[-2]
            max_delta = max(abs(cyc_loss[k] - prev[k]) for k in cyc_loss)
            if max_delta < args.plateau_eps:
                patience += 1
            else:
                patience = 0
            print(f"STAGE: plateau max_delta={max_delta:.4f} patience={patience}/{args.plateau_patience}")
            if patience >= args.plateau_patience:
                print(f"STAGE: early-stop: all cycle losses plateaued after {cyc} cycles")
                break

    # Validation examples (conclude from real samples, not just aggregates).
    dump_validation_examples(encoder, make_b, composer, ans_dec, qa_samples,
                             TOK, device, n_per_task=5)

    # ---- Final eval + isolation diagnostics ----
    print("STAGE: eval")
    acc, task_acc, n, gens = run_qa(encoder, make_b, composer, ans_dec,
                                    qa_samples, TOK, device)
    hist["qa_acc"].append(acc)
    for t, a in task_acc.items():
        hist["qa_task"].setdefault(t, []).append(a)
    print(f"\nCompositional exact-match accuracy: {acc:.3f} (n={n})")
    for t, a in task_acc.items():
        print(f"  {t}: {a:.3f}")
    print(f"STAGE: done qa_acc={acc:.3f}")

    # (b/c) isolation: is the I/O + head fine given the TRUE teacher state?
    oracle = diag.io_oracle_tests_with_decoder(
        encoder, decoder, composer, ans_dec, make_b, qa_samples, TOK, device)
    makeb = diag.make_b_recovery(encoder, make_b, qa_samples, TOK, device)
    hist_gen = diag.token_histogram(gens, TOK)
    print("STAGE: oracle " + json.dumps(oracle))
    print("STAGE: make_B " + json.dumps(makeb))
    print("STAGE: gen_hist " + json.dumps(hist_gen))
    print(f"  >> I-O/head oracle acc={oracle['oracle_answer_head_acc']} -> {oracle['interpretation']}")
    print(f"  >> generated output: {hist_gen['verdict']}")

    # save everything
    report = {
        "qa_accuracy": acc,
        "task_accuracy": task_acc,
        "n": n,
        "d_state": args.d_state,
        "data_diversity": div,
        "io_oracle": oracle,
        "make_B": makeb,
        "gen_histogram": hist_gen,
    }
    Path("modules_report.json").write_text(json.dumps(report, indent=2))
    Path("telemetry.json").write_text(json.dumps(
        {"config": vars(args), "history": hist, "final_report": report}, indent=2))
    diag.save_telemetry("diagnostics.txt", _render_diagnostics(div, oracle, makeb, hist_gen, hist, acc, task_acc))
    diag.render_curves(hist, "curves.png")
    print("Saved modules_report.json, telemetry.json, diagnostics.txt, curves.png")


def _render_diagnostics(div, oracle, makeb, hist_gen, hist, acc, task_acc) -> str:
    L = []
    L.append("=" * 70)
    L.append("DIAGNOSTICS: why did this run score the way it did?")
    L.append("=" * 70)
    L.append("\n[1] DATA DIVERSITY  (is the dataset too easy / biased?)")
    L.append(f"    QA samples={div['n_qa']}  unique answers={div['n_unique_answers']}")
    L.append(f"    majority-class baseline acc={div['majority_baseline_acc']}")
    L.append(f"    verdict: {div['verdict']}")
    L.append("\n[2] TRAINING HEALTH  (do losses move the right way?)")
    if hist["phase0_loss"]:
        L.append(f"    autoenc recon_loss: start={hist['phase0_loss'][0]:.3f} "
                 f"end={hist['phase0_loss'][-1]:.3f}")
    for name, vals in hist["phase1"].items():
        if vals:
            L.append(f"    {name}: start={vals[0]:.3f} end={vals[-1]:.3f} "
                     f"{'OK(down)' if vals[-1] < vals[0] else 'BAD(up)'}")
    L.append("\n[3] MATH / I-O  (does the architecture actually work?)")
    L.append(f"    autoenc reconstruction char-acc = {oracle['autoenc_recon_char_acc']}")
    L.append(f"    ORACLE answer-head acc (true teacher state) = {oracle['oracle_answer_head_acc']}")
    L.append(f"    composer reaches D_target MSE = {oracle['composer_D_mse']}")
    L.append(f"    make_B(A) recovery MSE = {makeb['make_B_mse']} ({makeb['verdict']})")
    L.append(f"    interpretation: {oracle['interpretation']}")
    L.append("\n[4] GENERATION OUTPUT  (what are we actually producing?)")
    L.append(f"    token fractions={hist_gen['fractions']}")
    L.append(f"    verdict: {hist_gen['verdict']}")
    L.append("\n[5] FINAL QA")
    L.append(f"    overall acc={acc:.3f}")
    for t, a in task_acc.items():
        L.append(f"      {t}: {a:.3f}")
    L.append("")
    return "\n".join(L)


def report_analyze():
    p = Path("telemetry.json")
    if not p.exists():
        print("telemetry.json not found (run first).")
        return
    data = json.loads(p.read_text())
    hist = data["history"]
    rep = data["final_report"]
    print("=== LAST RUN (from telemetry.json) ===")
    print(f"final QA acc={rep['qa_accuracy']:.3f}  task_acc={rep['task_accuracy']}")
    if hist.get("qa_acc"):
        print(f"QA accuracy over time: {['%.2f' % x for x in hist['qa_acc']]}")
    print("\nDATA:", rep["data_diversity"]["verdict"])
    print("I-O/HEAD ORACLE:", rep["io_oracle"]["interpretation"])
    print("GEN:", rep["gen_histogram"]["verdict"])


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/latent.py
"""Converged hybrid latent-state model (SSM think + FFN speak).

- One shared SSM with a `derive` switch (source | answer). Confidence is the
  transformed training loss; it loops (soft-gated in training) until loss<eps.
- A forward FFN (latent->tokens) with a complete head, derive-conditioned.
- Cross-mode FFN(L_src, derive=ANS)->Answer is trained so inference works.
- A token-by-token BaselineAR of similar size for the "latent vs tokens" test.

Synthetic random dataset: ~256 categorized vocab, multi-fact worlds rendered
as A=prose / B=json / C=question / D=answer. d_state is set from L*.
"""
import math, random
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------- vocab / dataset ----------------
def build_vocab():
    cats = {
        "name": [f"N{i}" for i in range(80)],
        "obj":  [f"O{i}" for i in range(80)],
        "verb": [f"V{i}" for i in range(32)],
        "pron": [f"P{i}" for i in range(24)],
        "loc":  [f"L{i}" for i in range(32)],
        "punct": [".", ",", '"', "'", "?", ":"],
        "data": ["{", "}", "JSON"],
    }
    syms = [s for v in cats.values() for s in v]
    special = ["<PAD>", "<BOS>", "<EOS>", "<DERIVE_SRC>", "<DERIVE_ANS>"]
    vocab = special + syms  # ~260 tokens
    return vocab, cats


class Tok:
    def __init__(self, vocab, cats):
        self.vocab = vocab
        self.cats = cats
        self.stoi = {s: i for i, s in enumerate(vocab)}
        self.itos = vocab
        self.pad = self.stoi["<PAD>"]
        self.bos = self.stoi["<BOS>"]
        self.eos = self.stoi["<EOS>"]
        self.der_src = self.stoi["<DERIVE_SRC>"]
        self.der_ans = self.stoi["<DERIVE_ANS>"]

    def enc(self, toks):
        return [self.stoi[t] for t in toks if t in self.stoi]

    def dec(self, ids):
        return [self.itos[i] for i in ids]


def gen_world(tok, rng, max_facts=4):
    k = rng.randint(1, max_facts)
    facts = []
    for _ in range(k):
        subj = rng.choice(tok.cats["name"])
        verb = rng.choice(tok.cats["verb"])
        obj = rng.choice(tok.cats["obj"] + tok.cats["loc"])
        facts.append((subj, verb, obj))
    source = [t for (s, v, o) in facts for t in (s, v, o, ".")]
    data = [t for (s, v, o) in facts for t in ("JSON", s, v, o)]
    qi = rng.randrange(k)
    qs, qv, qo = facts[qi]
    question = [qs, qv, "?"]
    answer = [qo]
    return {"source": source, "data": data, "question": question,
            "answer": answer, "k": k}


def compute_Lstar(cats, max_facts=4, bits_per_float=16):
    """Theoretical best latent size (information-theoretic)."""
    d_fact = (math.log2(len(cats["name"])) + math.log2(len(cats["verb"]))
              + math.log2(len(cats["obj"]) + len(cats["loc"])))
    bits = max_facts * d_fact + math.log2(2) + 4 + 4  # +switch+storyVariant+misc
    floats = math.ceil(bits / bits_per_float)
    return int(floats), round(bits, 1)


# ---------------- latent model: SSM think + FFN speak ----------------
class LatentModel(nn.Module):
    def __init__(self, vocab, d_emb=32, d_ctx=48, d_state=16, d_der=8,
                 d_hidden=64):
        super().__init__()
        self.vocab = len(vocab)
        self.d_emb = d_emb
        self.d_state = d_state
        self.d_der = d_der
        self.emb = nn.Embedding(self.vocab, d_emb, padding_idx=0)
        self.ctx_enc = nn.Linear(d_emb, d_ctx)
        self.tgt_enc = nn.Linear(d_emb, d_state)
        self.q_enc = nn.Linear(d_emb, d_emb)
        self.der_emb = nn.Embedding(2, d_der)
        self.ssm = nn.Sequential(nn.Linear(d_state + d_ctx + d_der, d_state),
                                 nn.Tanh())
        self.s0 = nn.Parameter(torch.zeros(d_state))
        self.gru = nn.GRU(d_emb + d_state + d_der + d_emb, d_hidden,
                          batch_first=True)
        self.out = nn.Linear(d_hidden, self.vocab)
        self.comp = nn.Linear(d_hidden, 1)

    def ctx(self, ids):
        return self.ctx_enc(self.emb(ids).mean(0))

    def target(self, ids):
        return self.tgt_enc(self.emb(ids).mean(0))

    def qvec(self, ids):
        return self.q_enc(self.emb(ids).mean(0))

    def ssm_loss(self, ctx, der_id, K, target, lam=0.1):
        der = self.der_emb(der_id)
        s = self.s0
        states, confs = [], []
        for _ in range(K):
            s = self.ssm(torch.cat([s, ctx, der]))
            states.append(s)
            confs.append(1.0 / (1.0 + F.mse_loss(s, target)))
        ws, prod = [], 1.0
        for c, o in zip(confs, [torch.ones_like(c) - c for c in confs]):
            ws.append(c * prod)
            prod = prod * o
        wsum = sum(ws) + prod
        ws = [w / wsum for w in ws]
        prod /= wsum
        s_eff = sum(w * st for w, st in zip(ws, states)) + prod * states[-1]
        loss = F.mse_loss(s_eff, target) + lam * sum(
            w * F.mse_loss(st, target) for w, st in zip(ws, states))
        return loss, s_eff

    def ssm_hard(self, ctx, der_id, K, target, tau=0.5):
        der = self.der_emb(der_id)
        s = self.s0
        for _ in range(K):
            s = self.ssm(torch.cat([s, ctx, der]))
            if (1.0 / (1.0 + F.mse_loss(s, target))).item() > tau:
                break
        return s

    def ffn_loss(self, s, der_id, q_ids, tgt_ids):
        if isinstance(tgt_ids, torch.Tensor):
            tgt_ids = tgt_ids.tolist()
        if isinstance(q_ids, torch.Tensor):
            q_ids = q_ids.tolist()
        der = self.der_emb(der_id)
        dev = self.emb.weight.device
        qv = self.qvec(torch.tensor(q_ids, device=dev))
        h = torch.zeros(1, 1, self.gru.hidden_size, device=dev)
        toks = [self.bos] + tgt_ids
        logits, comps = [], []
        for i in range(len(tgt_ids)):
            x = torch.cat([self.emb(torch.tensor(toks[i], device=dev)), s, der, qv]
                          ).unsqueeze(0).unsqueeze(0)
            out, h = self.gru(x, h)
            logits.append(self.out(out[0, 0]))
            comps.append(torch.sigmoid(self.comp(out[0, 0])))
        logits = torch.stack(logits)
        comps = torch.stack(comps).squeeze(1)
        ce = F.cross_entropy(logits, torch.tensor(tgt_ids, device=dev))
        tgt_comp = torch.zeros(len(tgt_ids), device=dev)
        tgt_comp[-1] = 1.0
        bce = F.binary_cross_entropy(comps, tgt_comp)
        return ce + bce

    def ffn_gen(self, s, der_id, q_ids, max_len=12, tau=0.5):
        der = self.der_emb(der_id)
        dev = self.emb.weight.device
        qv = self.qvec(torch.tensor(q_ids, device=dev))
        h = torch.zeros(1, 1, self.gru.hidden_size, device=dev)
        toks = [self.bos]
        out_ids = []
        for _ in range(max_len):
            x = torch.cat([self.emb(torch.tensor(toks[-1], device=dev)), s, der, qv]
                          ).unsqueeze(0).unsqueeze(0)
            out, h = self.gru(x, h)
            nid = int(self.out(out[0, 0]).argmax())
            out_ids.append(nid)
            if torch.sigmoid(self.comp(out[0, 0])).item() > tau or nid == self.eos:
                break
            toks.append(torch.tensor(nid, device=dev))
        return out_ids


# ---------------- baseline: token-by-token AR (no latent) ----------------
class BaselineAR(nn.Module):
    """Standard autoregressive decoder: given (source+question) tokens, it
    generates the answer token-by-token. No latent think-loop / compression.
    Comparable capacity to LatentModel for the 'latent vs tokens' test."""

    def __init__(self, vocab, d_emb=32, d_hidden=64):
        super().__init__()
        self.vocab = len(vocab)
        self.emb = nn.Embedding(self.vocab, d_emb, padding_idx=0)
        self.ctx_enc = nn.Linear(d_emb, d_hidden)
        self.gru = nn.GRU(d_emb, d_hidden, batch_first=True)
        self.out = nn.Linear(d_hidden, self.vocab)
        self.comp = nn.Linear(d_hidden, 1)

    def forward_loss(self, ctx_ids, q_ids, tgt_ids, bos):
        if isinstance(tgt_ids, torch.Tensor):
            tgt_ids = tgt_ids.tolist()
        if isinstance(q_ids, torch.Tensor):
            q_ids = q_ids.tolist()
        if isinstance(ctx_ids, torch.Tensor):
            ctx_ids = ctx_ids.tolist()
        dev = self.emb.weight.device
        c = self.emb(torch.tensor(ctx_ids + q_ids, device=dev)).mean(0)
        h = self.ctx_enc(c).unsqueeze(0).unsqueeze(0)
        toks = [bos] + tgt_ids
        logits, comps = [], []
        for i in range(len(tgt_ids)):
            x = self.emb(torch.tensor(toks[i], device=dev)).unsqueeze(0).unsqueeze(0)
            out, h = self.gru(x, h)
            logits.append(self.out(out[0, 0]))
            comps.append(torch.sigmoid(self.comp(out[0, 0])))
        logits = torch.stack(logits)
        comps = torch.stack(comps).squeeze(1)
        ce = F.cross_entropy(logits, torch.tensor(tgt_ids, device=dev))
        tgt_comp = torch.zeros(len(tgt_ids), device=dev)
        tgt_comp[-1] = 1.0
        bce = F.binary_cross_entropy(comps, tgt_comp)
        return ce + bce

    def generate(self, ctx_ids, q_ids, bos, max_len=12, tau=0.5):
        if isinstance(ctx_ids, torch.Tensor):
            ctx_ids = ctx_ids.tolist()
        if isinstance(q_ids, torch.Tensor):
            q_ids = q_ids.tolist()
        dev = self.emb.weight.device
        c = self.emb(torch.tensor(ctx_ids + q_ids, device=dev)).mean(0)
        h = self.ctx_enc(c).unsqueeze(0).unsqueeze(0)
        toks = [bos]
        out_ids = []
        for _ in range(max_len):
            x = self.emb(torch.tensor(toks[-1], device=dev)).unsqueeze(0).unsqueeze(0)
            out, h = self.gru(x, h)
            nid = int(self.out(out[0, 0]).argmax())
            out_ids.append(nid)
            if torch.sigmoid(self.comp(out[0, 0])).item() > tau or nid == self.eos:
                break
            toks.append(torch.tensor(nid, device=dev))
        return out_ids


In [ ]:
%%writefile train_converged.py
#!/usr/bin/env python3
"""Training + evaluation for the converged hybrid latent-state design.

Trains BOTH the latent model (SSM think + FFN speak) and an equal-capacity
token-by-token baseline (BaselineAR) on the same synthetic task, then reports
val exact-match QA for each -- the actual "latent thinking vs tokens" test.

Outputs modules_report.json (consumed by the notebook summary cell).

Local --quick = CPU sanity only. Real run on Kaggle (--device cuda).
"""
import argparse
import json
import random

import torch
import torch.optim as optim

from src.latent import (build_vocab, Tok, gen_world, compute_Lstar,
                         LatentModel, BaselineAR)


def main():
    print("=== train_converged main() entered (imports done) ===", flush=True)
    ap = argparse.ArgumentParser()
    ap.add_argument("--quick", action="store_true")
    ap.add_argument("--d_state", type=int, default=0)  # 0 => use L*
    ap.add_argument("--n_samples", type=int, default=5000)
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--K", type=int, default=4)
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--max_facts", type=int, default=4)
    ap.add_argument("--val_frac", type=float, default=0.1)
    args = ap.parse_args()
    if args.quick:
        args.n_samples = 400
        args.epochs = 3
        args.K = 4
        args.device = "cpu"

    vocab, cats = build_vocab()
    tok = Tok(vocab, cats)
    Lstar_floats, Lstar_bits = compute_Lstar(cats, args.max_facts)
    d_state = (args.d_state if args.d_state > 0
               else max(8, min(48, Lstar_floats + 8)))
    rng = random.Random(0)
    data = [gen_world(tok, rng, args.max_facts) for _ in range(args.n_samples)]
    random.Random(1).shuffle(data)
    nval = max(1, int(args.val_frac * len(data)))
    val, tr = data[:nval], data[nval:]

    dev = args.device
    latent = LatentModel(vocab, d_state=d_state).to(dev)
    latent.bos = tok.bos
    latent.eos = tok.eos
    baseline = BaselineAR(vocab).to(dev)
    baseline.bos = tok.bos
    baseline.eos = tok.eos
    optL = optim.Adam(latent.parameters(), lr=1e-3)
    optB = optim.Adam(baseline.parameters(), lr=1e-3)
    der_src = torch.tensor(0, device=dev)
    der_ans = torch.tensor(1, device=dev)

    print(f"vocab={len(vocab)} L*={Lstar_bits}b->{Lstar_floats}f "
          f"d_state={d_state} n_train={len(tr)} n_val={len(val)} "
          f"epochs={args.epochs} K={args.K} device={dev}")

    for ep in range(args.epochs):
        for s in tr:
            srcq = torch.tensor(tok.enc(s["source"] + s["question"]), device=dev)
            q = tok.enc(s["question"])
            a = tok.enc(s["answer"])
            qid = torch.tensor(q, device=dev)
            aid = torch.tensor(a, device=dev)
            # --- latent model ---
            c_src, t_src = latent.ctx(srcq), latent.target(srcq)
            c_ans, t_ans = latent.ctx(aid), latent.target(aid)
            l1, se = latent.ssm_loss(c_src, der_src, args.K, t_src)
            l2, ae = latent.ssm_loss(c_ans, der_ans, args.K, t_ans)
            l3 = latent.ffn_loss(se, der_src, qid, q)        # L_src -> question
            l4 = latent.ffn_loss(ae, der_ans, qid, a)        # L_ans -> answer
            l5 = latent.ffn_loss(se, der_ans, qid, a)        # L_src -> answer (cross-mode)
            optL.zero_grad()
            (l1 + l2 + l3 + l4 + l5).backward()
            optL.step()
            # --- baseline (token-by-token) ---
            optB.zero_grad()
            baseline.forward_loss(srcq.tolist(), q, a, baseline.bos).backward()
            optB.step()

        # --- validation ---
        accL = accB = n = 0
        for s in val:
            srcq = torch.tensor(tok.enc(s["source"] + s["question"]), device=dev)
            q = tok.enc(s["question"])
            a = tok.enc(s["answer"])
            c_src, t_src = latent.ctx(srcq), latent.target(srcq)
            s_src = latent.ssm_hard(c_src, der_src, args.K, t_src, tau=0.5)
            gen = latent.ffn_gen(s_src, der_ans, q, max_len=12, tau=0.5)
            if tok.dec(gen) == tok.dec(a):
                accL += 1
            bg = baseline.generate(srcq.tolist(), q, baseline.bos, max_len=12, tau=0.5)
            if tok.dec(bg) == tok.dec(a):
                accB += 1
            n += 1
        print(f"  epoch {ep} val latent={accL / n:.3f} baseline={accB / n:.3f}")

    report = {
        "design": "converged SSM(think)+FFN(speak) vs token-by-token baseline",
        "Lstar_bits": Lstar_bits,
        "Lstar_floats": Lstar_floats,
        "d_state": d_state,
        "vocab": len(vocab),
        "max_facts": args.max_facts,
        "n_train": len(tr),
        "n_val": len(val),
        "epochs": args.epochs,
        "val_latent_acc": accL / n,
        "val_baseline_acc": accB / n,
        "interpretation": (
            "latent wins" if accL > accB else
            "baseline wins" if accB > accL else "tie"),
    }
    with open("modules_report.json", "w") as f:
        json.dump(report, f, indent=1)
    print("REPORT " + json.dumps(report))


if __name__ == "__main__":
    main()


In [ ]:
# Run the modular training (now on disk). Each piece (token<->state, and the
# latent algebra make_B/make_A/continue/Answer_in_format_D) is trained with
# its OWN objective, in a curriculum: first the autoencoder ('output sane
# words'), then the latent ops. Output streams here so kaggle_run.py can
# watch STAGE: lines live. Outputs (modules_report.json) land in CWD.
import sys, os
cmd = (f"{sys.executable} -u train_converged.py --device cpu "
       f"--n_samples 5000 --d_state 0 --epochs 20 --max_facts 4")
print('STAGE: launch', cmd)
os.system(cmd)
print('NOTEBOOK DONE')


In [ ]:
# Show the generated report (also saved as modules_report.json).
report = Path('modules_report.json')
if report.exists():
    print(report.read_text())
else:
    print('modules_report.json not found (run may have failed).')
